# M27 · Jeux et distances entre lois — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m27_jeux_distances/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). Dans les parties A à D, donne les valeurs que rend le calcul, **sans les arrondir** : les décimales demandées sont celles de ta copie. « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m27_jeux_distances/03_notebook.ipynb`, créée par `python tools/start_chapter.py M27`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M27.1 à M27.10, M27.16 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M27.19 | Les valeurs d'un jeu en stratégies pures : maximin, minimax et coups prudents, points selles compris · `pure_strategy_values` | 🔨 | ★ | 15 |
|  | M27.20 | Les stratégies optimales et la valeur d'un jeu 2 × 2, avec ou sans point selle · `mixed_strategy_value_2x2` | 🔨 | ★★ | 20 |
|  | M27.21 | Les itérés de la descente-montée simultanée ou alternée sur un jeu bilinéaire xᵀAy · `descent_ascent_iterates` | 🔨 | ★★ | 20 |
|  | M27.22 | W₂ de deux gaussiennes en dimension 1, à partir des moyennes et des écarts-types · `w2_gaussians_1d` | 🔨 | ★ | 10 |
|  | M27.23 | `scipy.optimize.linprog` : le transport optimal entre deux lois discrètes de la droite comme programme linéaire, comparé au plan monotone et à l'aire entre les fonctions de répartition | 📦 | ★★ | 25 |
| B | M27.24 | Prédire la trajectoire de la descente-montée simultanée puis alternée sur xy, puis la tracer : spirale qui s'éloigne ou courbe fermée ? | 🔮 | ★ | 15 |
|  | M27.25 | Le biais d'une distance estimée : ‖x̄₁ − x̄₂‖² entre deux échantillons d'une même loi de ℝ³, selon leurs tailles, contre tr Σ₁/n₁ + tr Σ₂/n₂ | 🔬 | ★★ | 20 |
|  | M27.26 | Deux fonctions de répartition sur une même figure : l'aire entre elles (W₁) et leur plus grand écart vertical (KS), pour une translation puis une dilatation | 🎨 | ★ | 20 |
| C | M27.27 | Chasse à l'erreur : une stratégie d'indifférence calculée sur ses propres gains, W₂ de deux gaussiennes avec des variances, des échantillons appariés sans être triés | 🐛 | ★ | 25 |
|  | M27.28 | Un test de propriété : la valeur d'un jeu 2 × 2 tiré au hasard est entre maximin et minimax, et devient v + c pour G + c et k v pour k G (k > 0) | 🛠️ | ★ | 20 |
| D | M27.29 | Défi : la gaussienne la plus proche d'une loi exponentielle au sens de W₂, et cette distance, à 10⁻³ près | 🏆 | ★★★ | 30 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M27")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : dominance et points selles, un jeu 2 × 2 par indifférence, des équilibres de Nash et des allumettes, la descente-montée sur $xy$, une probabilité a posteriori et trois log-loss, JS de deux lois uniformes, un plan de transport, $W_1$ et $W_2$ en dimension 1, une distance de Fréchet, une dérive, et un pont vers la convention des bibliothèques.
- Écrire les quatre fonctions de `mymath/distances.py` (`pure_strategy_values`, `mixed_strategy_value_2x2`, `descent_ascent_iterates`, `w2_gaussians_1d`) et les tester.
- Résoudre un transport optimal par `linprog`, prédire puis tracer une descente-montée, simuler le biais d'une distance estimée, dessiner $W_1$ et KS, corriger trois erreurs classiques, écrire un test de propriété, et relever un défi.

**Rappel express.** Maximin $\max_i\min_j g_{ij}$ $\le$ minimax $\min_j\max_i g_{ij}$, égalité si et seulement s'il y a un point selle. Jeu $2 \times 2$ sans point selle : $\kappa = a - b - c + d$, $p^\star = \frac{d - c}{\kappa}$, $q^\star = \frac{d - b}{\kappa}$, $v = \frac{ad - bc}{\kappa}$. Descente-montée sur $xy$ : simultanée, valeurs propres $1 \pm i\eta$ de module $\sqrt{1 + \eta^2}$ ; alternée, déterminant 1 et trace $2 - \eta^2$. Transport : $W_p^p = \min\sum\gamma_{ij}\lvert x_i - y_j\rvert^p$ sur les plans de marges $\mathbf{a}$ et $\mathbf{b}$ ; sur la droite, $W_1 = \int\lvert F - G\rvert$ et $W_p^p = \int_0^1\lvert F^{-1} - G^{-1}\rvert^p$ ; $W_2^2 = (\mu_1 - \mu_2)^2 + (\sigma_1 - \sigma_2)^2$ pour deux gaussiennes. Biais : $\mathbb{E}\lVert\bar{\mathbf{x}}_1 - \bar{\mathbf{x}}_2\rVert^2 = \lVert\boldsymbol{\mu}_1 - \boldsymbol{\mu}_2\rVert^2 + \frac{\operatorname{tr}\boldsymbol{\Sigma}_1}{n_1} + \frac{\operatorname{tr}\boldsymbol{\Sigma}_2}{n_2}$. Dérive : $\mathrm{KS} = \sup\lvert F - G\rvert$, sans unité ; $W_1$ dans l'unité de la variable.

## Partie 0 · Vérifier tes exercices papier (✏️ M27.1 à M27.10 ; 🔗 M27.16)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question (« exact value » ou « a number » sans décimales : tous les chiffres, ces calculs tombent juste) ;
- une liste `[0.2, -1]` ; une matrice comme la liste de ses **lignes**, `[[1, 0], [-2, 3]]` ;
- une fraction irréductible entre guillemets, `"3/7"`, quand la question le dit ;
- `True` ou `False` pour un vrai ou faux.

Les réponses pas encore remplies affichent ⏳.

In [ ]:
# Part 0: a fraction asked as a text "p/q" must be a text (a float such as 5/13 is only close to it)
FRACTIONS_0 = ['M27.2a', 'M27.2b', 'M27.3c', 'M27.3d', 'M27.8a']


def check_paper_0(ex_id, answer):
    """wbm.check(ex_id, answer); for a fraction asked as a text "p/q", a number is refused (❌)."""
    if ex_id in FRACTIONS_0 and answer is not ... and answer is not None and not isinstance(answer, str):
        print(f"❌ Ex {ex_id} : la question demande une fraction irréductible entre guillemets, comme \"3/7\" ; reçu "
              f"{answer!r} ({type(answer).__name__}) : écris la fraction exacte, en texte.")
        return None
    return wbm.check(ex_id, answer)

**Ex M27.1 — Deux enseignes se partagent un marché : dominance, points selles, maximin et minimax**

In [ ]:
# Ex M27.1: the values you found on paper
answer_M27_1a = ...  # a) a list of 2 numbers: the maximin, then the minimax
answer_M27_1b = ...  # b) a list of pairs [row, column], numbered from 1, like [[1, 2], [3, 1]]
answer_M27_1c = ...  # c) True or False
answer_M27_1d = ...  # d) a matrix, as the list of its rows

for letter, answer in zip("abcd", [answer_M27_1a, answer_M27_1b, answer_M27_1c, answer_M27_1d]):
    check_paper_0(f"M27.1{letter}", answer)

**Ex M27.2 — Tirs au but : les stratégies d'indifférence, la valeur, et une stratégie qui s'en écarte**

In [ ]:
# Ex M27.2: the values you found on paper
answer_M27_2a = ...  # a) an irreducible fraction, as text "p/q"
answer_M27_2b = ...  # b) an irreducible fraction, as text "p/q"
answer_M27_2c = ...  # c) a number (2 decimals)
answer_M27_2d = ...  # d) a number (2 decimals)
answer_M27_2e = ...  # e) a number (2 decimals)

for letter, answer in zip("abcde", [answer_M27_2a, answer_M27_2b, answer_M27_2c, answer_M27_2d, answer_M27_2e]):
    check_paper_0(f"M27.2{letter}", answer)

**Ex M27.3 — Un dilemme du prisonnier, un jeu de coordination asymétrique, et des allumettes**

In [ ]:
# Ex M27.3: the values you found on paper
answer_M27_3a = ...  # a) a list of 2 numbers: the gains (rows, columns) at the equilibrium
answer_M27_3b = ...  # b) True or False
answer_M27_3c = ...  # c) an irreducible fraction, as text "p/q"
answer_M27_3d = ...  # d) an irreducible fraction, as text "p/q"
answer_M27_3e = ...  # e) a list of 2 numbers (3 decimals)
answer_M27_3f = ...  # f) a list of integers, in increasing order
answer_M27_3g = ...  # g) an integer

for letter, answer in zip("abcdefg", [answer_M27_3a, answer_M27_3b, answer_M27_3c, answer_M27_3d, answer_M27_3e, answer_M27_3f, answer_M27_3g]):
    check_paper_0(f"M27.3{letter}", answer)

**Ex M27.4 — Descente-montée sur xy : modules, angles, et deux pas différents**

In [ ]:
# Ex M27.4: the values you found on paper
answer_M27_4a = ...  # a) a number (4 decimals)
answer_M27_4b = ...  # b) a number (1 decimal)
answer_M27_4c = ...  # c) an integer
answer_M27_4d = ...  # d) a number (1 decimal)
answer_M27_4e = ...  # e) a list of 2 numbers (4 decimals)
answer_M27_4f = ...  # f) a number (2 decimals)

for letter, answer in zip("abcdef", [answer_M27_4a, answer_M27_4b, answer_M27_4c, answer_M27_4d, answer_M27_4e, answer_M27_4f]):
    check_paper_0(f"M27.4{letter}", answer)

**Ex M27.5 — Deux fournisseurs, un défaut sur quatre niveaux : a posteriori et log-loss de trois prédicteurs**

In [ ]:
# Ex M27.5: the values you found on paper
answer_M27_5a = ...  # a) a number (3 decimals)
answer_M27_5b = ...  # b) a list of 4 numbers (3 decimals)
answer_M27_5c = ...  # c) a number (4 decimals)
answer_M27_5d = ...  # d) a number (4 decimals)
answer_M27_5e = ...  # e) a number (4 decimals)

for letter, answer in zip("abcde", [answer_M27_5a, answer_M27_5b, answer_M27_5c, answer_M27_5d, answer_M27_5e]):
    check_paper_0(f"M27.5{letter}", answer)

**Ex M27.6 — Deux lois uniformes de même largeur, translatées : JS, KL et W₁**

In [ ]:
# Ex M27.6: the values you found on paper
answer_M27_6a = ...  # a) a number (3 decimals)
answer_M27_6b = ...  # b) a number (3 decimals)
answer_M27_6c = ...  # c) a list of 2 numbers (3 decimals)
answer_M27_6d = ...  # d) True or False
answer_M27_6e = ...  # e) a list of 2 numbers

for letter, answer in zip("abcde", [answer_M27_6a, answer_M27_6b, answer_M27_6c, answer_M27_6d, answer_M27_6e]):
    check_paper_0(f"M27.6{letter}", answer)

**Ex M27.7 — Trois dépôts et trois magasins : plans, coûts, couplage indépendant, coin nord-ouest, W₁**

In [ ]:
# Ex M27.7: the values you found on paper
answer_M27_7a = ...  # a) True or False
answer_M27_7b = ...  # b) a number
answer_M27_7c = ...  # c) a number
answer_M27_7d = ...  # d) a 3 × 3 matrix
answer_M27_7e = ...  # e) a number (2 decimals)

for letter, answer in zip("abcde", [answer_M27_7a, answer_M27_7b, answer_M27_7c, answer_M27_7d, answer_M27_7e]):
    check_paper_0(f"M27.7{letter}", answer)

**Ex M27.8 — W₁ et W₂ de deux lois uniformes, puis W₂ de deux gaussiennes**

In [ ]:
# Ex M27.8: the values you found on paper
answer_M27_8a = ...  # a) an irreducible fraction, as text "p/q"
answer_M27_8b = ...  # b) a number (4 decimals)
answer_M27_8c = ...  # c) a number (4 decimals)
answer_M27_8d = ...  # d) a number (4 decimals)

for letter, answer in zip("abcd", [answer_M27_8a, answer_M27_8b, answer_M27_8c, answer_M27_8d]):
    check_paper_0(f"M27.8{letter}", answer)

**Ex M27.9 — La distance de Fréchet de deux gaussiennes du plan aux vecteurs propres communs**

In [ ]:
# Ex M27.9: the values you found on paper
answer_M27_9a = ...  # a) a list of 2 lists of 2 numbers
answer_M27_9b = ...  # b) a number
answer_M27_9c = ...  # c) a number
answer_M27_9d = ...  # d) a number
answer_M27_9e = ...  # e) a number (2 decimals)

for letter, answer in zip("abcde", [answer_M27_9a, answer_M27_9b, answer_M27_9c, answer_M27_9d, answer_M27_9e]):
    check_paper_0(f"M27.9{letter}", answer)

**Ex M27.10 — Une dérive : PSI, JS et leur approximation, puis KS et W₁ entre deux petits échantillons**

In [ ]:
# Ex M27.10: the values you found on paper
answer_M27_10a = ...  # a) a number (4 decimals)
answer_M27_10b = ...  # b) a number (5 decimals)
answer_M27_10c = ...  # c) a number (4 decimals)
answer_M27_10d = ...  # d) a number (2 decimals)
answer_M27_10e = ...  # e) a number (2 decimals)
answer_M27_10f = ...  # f) a list of 2 numbers (3 decimals)

for letter, answer in zip("abcdef", [answer_M27_10a, answer_M27_10b, answer_M27_10c, answer_M27_10d, answer_M27_10e, answer_M27_10f]):
    check_paper_0(f"M27.10{letter}", answer)

**Ex M27.16 — Pont : la distance de Fréchet en dimension 1, avec des écarts-types ou des covariances**

In [ ]:
# Ex M27.16: the values you found on paper
answer_M27_16a = ...  # a) a number (exact value)
answer_M27_16b = ...  # b) a number (4 decimals)
answer_M27_16c = ...  # c) a number (1 decimal)
answer_M27_16d = ...  # d) a number (exact value)
answer_M27_16e = ...  # e) a number (4 decimals)

for letter, answer in zip("abcde", [answer_M27_16a, answer_M27_16b, answer_M27_16c, answer_M27_16d, answer_M27_16e]):
    check_paper_0(f"M27.16{letter}", answer)

## Partie A · Ta librairie : les valeurs d'un jeu, ses stratégies optimales, la descente-montée et W₂ ; puis le transport optimal par SciPy

Fiche §1, §2, §4, §7 et §8. Les quatre fonctions de `mymath/distances.py`, leurs tests, et un usage de chacune sur les données des exercices papier ; puis le transport optimal comme programme linéaire. La cellule de données ci-dessous recopie les données des exercices papier.

In [ ]:
# Tools of the notebook (parts A to D)
import math
import numbers
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
from scipy import integrate, optimize, stats

from wbm.errors import MymathMissingError
from wbm.nbtools import StopCheck, checked, drawn, filled, fr, own_code, pair, run_pytest_group, same, verdict

TEST_M27 = "tests/test_m27_distances.py"            # the tests of mymath/distances.py
REF = wbm.load_mymath("ref", alias="mymath_ref")   # the reference: the data cells and the checks never use your code
ref_distances = REF.distances
REF_DISTANCES = {f"ref_distances.{name}": getattr(ref_distances, name)    # a check stops if your code calls one of them
                 for name in ("pure_strategy_values", "mixed_strategy_value_2x2", "descent_ascent_iterates",
                              "w2_gaussians_1d")}


def shown(value, digits=4):
    """A short text for a result of the learner's code (the check messages): numbers rounded, as lists."""
    if value is None:
        return "None"
    try:
        if isinstance(value, tuple):
            return "(" + ", ".join(str(np.round(np.asarray(v, dtype=float), digits).tolist()) for v in value) + ")"
        return str(np.round(np.asarray(value, dtype=float), digits).tolist())
    except (TypeError, ValueError):
        return repr(value)[:120]


def check_answer(ex_id, value, hint=None):
    """wbm.check(ex_id, value, computed=True). For a list that holds True or False, the message gives only how many
    elements are right, not the place of the first wrong one (it would let you guess the booleans one by one).
    hint(value), when given, may return a targeted « Piste » for a wrong answer (None otherwise)."""
    holds_bool = isinstance(value, (list, tuple)) and any(isinstance(v, (bool, np.bool_)) for v in value)
    if not holds_bool and hint is None:
        return wbm.check(ex_id, value, computed=True)
    result = wbm.check(ex_id, value, computed=True, quiet=True)
    message = result.message
    if holds_bool:
        for cut in (" Le premier faux est le ", " Premier élément faux à l'indice "):
            message = message.split(cut)[0]
    if hint is not None and result.status not in ("correct", "pending", "unknown"):
        try:
            extra = hint(value)
        except Exception:  # noqa: BLE001 - an odd answer: no targeted hint
            extra = None
        if extra:
            message = f"{message} Piste : {extra}"
    icon = {"correct": "✅", "pending": "⏳", "unknown": "❓"}.get(result.status, "❌")
    print(f"{icon} Ex {ex_id} : {message}")
    return result


def check_rounding(ex_id, label, value, limit, wrong=()):
    """A number that must be a rounding error (0 <= value < limit): ✅, ⏳ or ❌. wrong: pairs (text, number), a classic
    wrong number (to 1e-6 relative) that gets its own text."""
    if not filled(value):
        print(f"⏳ Ex {ex_id} : {label} : pas encore fait.")
        return None
    try:
        number = float(value)
    except (TypeError, ValueError):
        print(f"❌ Ex {ex_id} : {label} : attendu un nombre, reçu {value!r}.")
        return False
    if 0 <= number < limit:
        print(f"✅ Ex {ex_id} : {label} : {number:.1e}, une erreur d'arrondi.")
        return True
    for text, target in wrong:
        if math.isclose(number, target, rel_tol=1e-6):
            print(f"❌ Ex {ex_id} : {label} : {text}.")
            return False
    print(f"❌ Ex {ex_id} : {label} : attendu un écart de l'ordre des arrondis (positif, moins de {limit:.0e}), reçu "
          f"{number!r}.")
    return False


def reporter(results, stop=True):
    """A function report(ok, success, failure) that prints one ✅ or ❌ line, keeps the verdict in results, and (stop)
    ends the check at the first ❌ (StopCheck, caught by checked)."""
    def report(ok, success, failure):
        results.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))
        if not ok and stop:
            raise StopCheck
    return report


def matches(got, want, rtol=1e-9, atol=1e-12):
    """True when got is an array of numbers of the shape of want and equal to it up to rounding (no error for an odd
    got): to recognize a classic wrong result before the comparison."""
    try:
        arr = np.asarray(got, dtype=float)
    except (TypeError, ValueError):
        return False
    want = np.asarray(want, dtype=float)
    return arr.shape == want.shape and bool(np.allclose(arr, want, rtol=rtol, atol=atol))


def as_floats(value):
    """value as a float array, or None when it is not an array of numbers."""
    try:
        return np.asarray(value, dtype=float)
    except (TypeError, ValueError):
        return None


def line_points(line):
    """The (x, y) points of a line of matplotlib, as two float arrays."""
    return np.asarray(line.get_xdata(), dtype=float), np.asarray(line.get_ydata(), dtype=float)


def through(ax, xs, ys, rtol=1e-7, atol=1e-12):
    """True when one line of ax passes by all the points (xs, ys), in this order."""
    xs, ys = np.asarray(xs, dtype=float), np.asarray(ys, dtype=float)
    for line in ax.lines:
        x, y = line_points(line)
        if (x.shape == xs.shape and y.shape == ys.shape and np.allclose(x, xs, rtol=rtol, atol=atol)
                and np.allclose(y, ys, rtol=rtol, atol=atol)):
            return True
    return False


def drawn_points(ax, marker_only=False):
    """All the points drawn in ax: those of its lines (marker_only: lines with a marker only) and of its scatters."""
    points = []
    for line in ax.lines:
        if marker_only and line.get_marker() in (None, "", " ", "None", "none"):
            continue
        x, y = line_points(line)
        points += list(zip(x.ravel(), y.ravel()))
    for collection in ax.collections:
        offsets = np.asarray(collection.get_offsets(), dtype=float)
        if offsets.ndim == 2 and offsets.shape[1] == 2 and type(collection).__name__ == "PathCollection":
            points += [tuple(p) for p in offsets]
    return points


def points_at(ax, xs, ys, rtol=1e-7, atol=1e-12, marker_only=False):
    """True when every point (xs[k], ys[k]) is drawn in ax (a line with or without markers, or a scatter)."""
    points = np.array(drawn_points(ax, marker_only) or [(np.nan, np.nan)], dtype=float)
    return all(np.any(np.isclose(points[:, 0], x, rtol=rtol, atol=atol) & np.isclose(points[:, 1], y, rtol=rtol, atol=atol))
               for x, y in zip(np.atleast_1d(xs), np.atleast_1d(ys)))


def follows(ax, f, lo, hi, rtol=1e-6):
    """True when one line of ax has at least two points with y = f(x), its x covering [lo, hi]."""
    for line in ax.lines:
        x, y = line_points(line)
        if (x.size >= 2 and np.all(np.isfinite(x)) and x.min() <= lo * (1 + 1e-9) and x.max() >= hi * (1 - 1e-9)
                and np.allclose(y, f(x), rtol=rtol, atol=0)):
            return True
    return False


# The data of the paper exercises (02_exercices.md), the strategies numbered from 0 as in mymath
G_MARKET = np.array([[5.0, 2, 6, 1], [4, 3, 7, 3], [0, 1, 4, 0]])     # M27.1: rows A1, A2, A3; columns B1 to B4
G_PENALTY = np.array([[55.0, 95], [85, 60]])                           # M27.2: shot left, right; keeper left, right (%)
DEPOTS_KM, DEPOTS_T = np.array([1.0, 4, 6]), np.array([20.0, 50, 30])  # M27.7: the depots (km) and their stocks (t)
STORES_KM, STORES_T = np.array([2.0, 3, 8]), np.array([40.0, 35, 25])  # the stores (km) and their orders (t)
BAGS_BEFORE, BAGS_AFTER = (20.0, 1.5), (21.2, 1.0)                     # M27.8: (mean, standard deviation) in g
SHAFTS_BEFORE, SHAFTS_AFTER = (50.30, 0.12), (50.02, 0.05)             # M27.16: (mean, standard deviation) in mm

### Ex M27.19 — Les valeurs d'un jeu en stratégies pures : maximin, minimax et coups prudents, points selles compris · `pure_strategy_values` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer le maximin, le minimax et les coups qui les donnent, puis voir qu'ils forment un point selle quand il en existe un, et seulement alors.  
**Prérequis :** fiche §1 ; Ex M27.1 et M27.2 · **mymath :** `distances.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M26). Ouvre ta copie `mon_travail/mymath/distances.py`, lis la docstring du module (ses conventions, et ce qui est permis) puis celle de la fonction, remplace chaque `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Dans le notebook, appelle tes fonctions par `mymath.distances.<nom>` (par exemple `mymath.distances.pure_strategy_values(G)`), ou donne-leur un nom court (`pure_strategy_values = mymath.distances.pure_strategy_values`) dans une cellule exécutée après le rechargement. ⚠️ Le squelette n'importe rien : ajoute en tête du fichier `import numpy as np` (et `import math`, `import numbers` si tu t'en sers). NumPy et le module `math` sont permis, mais rien de `np.linalg` et aucune fonction de SciPy (`scipy.optimize` résout un jeu) : les tests comparent tes résultats à ceux de `scipy.optimize.linprog`, de `np.linalg.matrix_power` et de `scipy.integrate.quad`, et un test lit ton module. Rends des nombres `float`, des indices `int` et des itérés en tableaux NumPy de flottants, et ne modifie jamais les tableaux reçus. Hors de `mymath`, le notebook peut se servir de tout NumPy et de SciPy.

Écris `pure_strategy_values(G)` (fiche §1), en suivant sa docstring : contrôle `G` (un tableau 2-D, non vide, de nombres finis ; sinon une `ValueError`) ; calcule le minimum de chaque ligne (`axis=1`) et le maximum de chaque colonne (`axis=0`), puis le maximin, le minimax, et les plus petits indices `row` et `col` où ils sont atteints (`np.argmax` et `np.argmin` rendent le premier en cas d'égalité) ; rends `(maximin, minimax, row, col)`, deux `float` et deux `int` Python.

**L'usage.** La cellule de données de la partie A donne le marché de M27.1 (`G_MARKET`) et les tirs au but de M27.2 (`G_PENALTY`). Avec ta fonction :
- a) le marché (un tuple de 4, tel que rendu), à comparer à tes réponses a) et b) de M27.1 (les indices partent de 0) ;
- b) le même jeu vu par l'enseigne B, `-G_MARKET.T` : ses lignes sont les stratégies de B, qui maximise ce que A perd (un tuple de 4, tel que rendu) ; dans ta copie, dis pourquoi le maximin de `-G.T` est l'opposé du minimax de `G` ;
- c) les tirs au but, et vrai ou faux : `(row, col)` y est un point selle (une liste de 5 : les 4 valeurs rendues, puis le booléen).

Ce qu'il faut observer : la fonction rend un point selle quand il en existe un ; sans point selle, `row` et `col` ne sont que des coups prudents, que l'adversaire exploite s'il les connaît.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M27")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M27, "test_pure_strategy_values_ or test_module_uses_")

**L'usage** : le marché, le même jeu vu par B, et les tirs au but.

In [ ]:
answer_M27_19a = answer_M27_19b = answer_M27_19c = ...
with wbm.attempt("M27.19"):          # if you call pure_strategy_values here: ⏳ while it is not written
    answer_M27_19a = ...  # a) a tuple of 4, as returned: the market G_MARKET
    answer_M27_19b = ...  # b) a tuple of 4, as returned: the same game seen by B, -G_MARKET.T
    answer_M27_19c = ...  # c) a list of 5: the 4 values returned for G_PENALTY, then True or False: (row, col) is a saddle point there

In [ ]:
for letter, answer in zip("abc", [answer_M27_19a, answer_M27_19b, answer_M27_19c]):
    check_answer(f"M27.19{letter}", answer)

### Ex M27.20 — Les stratégies optimales et la valeur d'un jeu 2 × 2, avec ou sans point selle · `mixed_strategy_value_2x2` 🔨 ★★ ⏱️ 20 min
**Objectif :** programmer les stratégies optimales d'un jeu 2 × 2, pures s'il y a un point selle, d'indifférence sinon, et voir que p* et q* garantissent la même valeur.  
**Prérequis :** fiche §2 ; Ex M27.2 et M27.19 · **mymath :** `distances.py` · **Parcours :** C, E

> Même mode d'emploi qu'en M27.19 : écris la fonction dans ta copie `mon_travail/mymath/distances.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `mixed_strategy_value_2x2(G)` (fiche §2), en suivant sa docstring : contrôle la forme $(2, 2)$, puis appelle ta `pure_strategy_values` ; s'il y a un point selle (`maximin == minimax`, une comparaison exacte), rends les stratégies pures, `p = 1.0` si `row == 0`, sinon `p = 0.0`, de même pour `q` avec `col`, et `value = maximin` ; sinon, rends les trois formules d'indifférence de la fiche, $p^\star = \frac{d - c}{\kappa}$, $q^\star = \frac{d - b}{\kappa}$, $v = \frac{ad - bc}{\kappa}$, en `float` Python.

**L'usage.** Avec ta fonction :
- a) les tirs au but de M27.2 (`G_PENALTY`) (un tuple de 3, tel que rendu), à comparer à tes réponses a) à c) ;
- b) pour $\mathbf{G} = \begin{pmatrix} 3 & 5 \\ 1 & 2 \end{pmatrix}$ : ce que donneraient les formules de $p^\star$ et de $q^\star$, puis ce que rend ta fonction (une liste de 5 nombres : $p$ et $q$ par les formules, puis les trois nombres rendus) ;
- c) pour les tirs au but, le plus petit gain espéré de $p^\star$ contre les deux plongeons, $\min_j(\mathbf{p}^{\star\top}\mathbf{G})_j$, puis le plus grand de $q^\star$ contre les deux tirs, $\max_i(\mathbf{G}\mathbf{q}^\star)_i$, avec $\mathbf{p}^\star = (p^\star, 1 - p^\star)$ et $\mathbf{q}^\star = (q^\star, 1 - q^\star)$ (une liste de 2 nombres).

Ce qu'il faut observer : avec un point selle, les formules d'indifférence sortent de $[0, 1]$ ; sans point selle, $p^\star$ et $q^\star$ garantissent la même valeur, chacun de son côté.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M27")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M27, "test_mixed_strategy_value_2x2_")

**L'usage** : les tirs au but, un jeu à point selle, et les deux garanties.

In [ ]:
G_SADDLE_20 = np.array([[3.0, 5.0], [1.0, 2.0]])
answer_M27_20a = answer_M27_20b = answer_M27_20c = ...
with wbm.attempt("M27.20"):          # if you call mixed_strategy_value_2x2 here: ⏳ while it is not written
    answer_M27_20a = ...  # a) a tuple of 3, as returned: (p, q, value) for G_PENALTY
    answer_M27_20b = ...  # b) a list of 5 numbers: p and q by the indifference formulas for G_SADDLE_20, then the 3 numbers your function returns
    answer_M27_20c = ...  # c) a list of 2 numbers, as computed: min_j (p*ᵀ G)_j, then max_i (G q*)_i, for G_PENALTY

In [ ]:
for letter, answer in zip("abc", [answer_M27_20a, answer_M27_20b, answer_M27_20c]):
    check_answer(f"M27.20{letter}", answer)

### Ex M27.21 — Les itérés de la descente-montée simultanée ou alternée sur un jeu bilinéaire xᵀAy · `descent_ascent_iterates` 🔨 ★★ ⏱️ 20 min
**Objectif :** programmer les deux descentes-montées, puis vérifier aux arrondis près les deux propriétés de la fiche : l'une éloigne de l'équilibre, l'autre garde une quantité constante.  
**Prérequis :** fiche §4 ; M22 ; Ex M27.4 · **mymath :** `distances.py` · **Parcours :** C

> Même mode d'emploi qu'en M27.19 : écris la fonction dans ta copie `mon_travail/mymath/distances.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `descent_ascent_iterates(A, x0, y0, eta, n_steps=10, alternating=False)` (fiche §4), en suivant sa docstring : les itérés de la descente-montée sur $f(\mathbf{x}, \mathbf{y}) = \mathbf{x}^\top\mathbf{A}\mathbf{y}$, où le joueur $\mathbf{x}$ (les lignes de $\mathbf{A}$) **minimise**, comme dans MML (7.23). Contrôle `A` (2-D, non vide, de nombres finis), `x0` et `y0` (des tableaux **1-D** de $n$ et de $m$ coordonnées : `np.array([1.0])`, pas `np.array([[1.0]])`), `eta` (un nombre fini $> 0$, pas un booléen) et `n_steps` (un entier $\ge 0$, ni un booléen ni un flottant) ; prépare deux tableaux de formes $(N + 1, n)$ et $(N + 1, m)$, de lignes 0 les départs ; à chaque pas, calcule le nouveau $\mathbf{x}$ à partir de l'ancien $\mathbf{y}$, puis le nouveau $\mathbf{y}$ à partir de l'ancien $\mathbf{x}$ (simultané) ou du nouveau (alterné) ; rends `(xs, ys)`.

**L'usage.** Sur $xy$ (`A = [[1.0]]`, depuis $x_0 = 1$ et $y_0 = 0$, avec $\eta = 0{,}6$), avec ta fonction :
- a) en simultané, la distance à l'équilibre $\sqrt{x_k^2 + y_k^2}$ après 4 pas, puis après 5 pas (une liste de 2 nombres), à comparer à ta réponse c) de M27.4 ;
- b) en alterné, sur 200 pas : le plus grand écart $\lvert Q(x_k, y_k) - Q(x_0, y_0)\rvert$, avec $Q(x, y) = x^2 + y^2 - \eta xy$, et vrai ou faux, la distance à l'équilibre ne dépasse jamais $1/\sqrt{0{,}7}$, le demi-grand axe de l'ellipse $Q = 1$ (une liste de 2 : le nombre, tel que calculé, puis le booléen) ;
- c) sur le jeu de matrice $\mathbf{A} = \begin{pmatrix} 1 & -2 & 0 \\ 0{,}5 & 1 & -1 \end{pmatrix}$ (`A_GAME_21` ; $\mathbf{x} \in \mathbb{R}^2$, $\mathbf{y} \in \mathbb{R}^3$), avec $\eta = 0{,}3$, depuis $\mathbf{x}_0 = (1, 0)$ et $\mathbf{y}_0 = (0, 1, 0)$ : vrai ou faux, en simultané, $\lVert\mathbf{x}_k\rVert^2 + \lVert\mathbf{y}_k\rVert^2$ ne diminue jamais sur 50 pas ; puis, en alterné, le plus grand écart de $\lVert\mathbf{x}_k\rVert^2 + \lVert\mathbf{y}_k\rVert^2 - \eta\,\mathbf{x}_k^\top\mathbf{A}\mathbf{y}_k$ à sa valeur de départ sur 100 pas (une liste de 2 : le booléen, puis le nombre, tel que calculé).

La vérification des deux écarts attend une erreur d'arrondi (moins de $10^{-12}$). Ce qu'il faut observer : les deux propriétés de la fiche, vérifiées aux arrondis près, l'une qui éloigne de l'équilibre, l'autre qui garde une quantité constante.

In [ ]:
A_GAME_21 = np.array([[1.0, -2.0, 0.0], [0.5, 1.0, -1.0]])   # c): x in R², y in R³

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M27")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M27, "test_descent_ascent_iterates_")

**L'usage** : le jeu $xy$ en simultané et en alterné, puis un jeu de $\mathbb{R}^2 \times \mathbb{R}^3$.

In [ ]:
answer_M27_21a = answer_M27_21b = answer_M27_21c = ...
with wbm.attempt("M27.21"):          # if you call descent_ascent_iterates here: ⏳ while it is not written
    answer_M27_21a = ...  # a) a list of 2 numbers, as computed: the distance to (0, 0) after 4, then after 5 simultaneous steps (xy, η = 0.6, from (1, 0))
    answer_M27_21b = ...  # b) a list of 2: the largest |Q(x_k, y_k) − Q(x_0, y_0)| over 200 alternating steps (as computed), then True or False: the distance never exceeds 1/√0.7
    answer_M27_21c = ...  # c) a list of 2: True or False: in simultaneous steps, ‖x‖² + ‖y‖² never decreases over 50 steps (A_GAME_21, η = 0.3); then the largest gap of ‖x‖² + ‖y‖² − η xᵀAy to its start over 100 alternating steps (as computed)

In [ ]:
check_answer("M27.21a", answer_M27_21a)

WRONG_C21B = [("c'est l'écart des itérés simultanés : la quantité n'est constante que pour le pas alterné "
  '(alternating=True)',
  4.2355181205062675e+26),
 ('Q = ‖x‖² + ‖y‖² − η xᵀAy : le signe du terme croisé compte', 0.8571312078673605)]   # (message, value) of two classic wrong deviations of Q on xy
WRONG_C21C = [("c'est l'écart des itérés simultanés : la quantité n'est constante que pour le pas alterné "
  '(alternating=True)',
  1.237426154211831e+18),
 ('Q = ‖x‖² + ‖y‖² − η xᵀAy : le signe du terme croisé compte', 3.855623964107798)]   # the same on the game A_GAME_21

with checked("M27.21"):
    if not filled(answer_M27_21b):
        print("⏳ Ex M27.21b : pas encore fait.")
    elif not isinstance(answer_M27_21b, (list, tuple)) or len(answer_M27_21b) != 2:
        print("❌ Ex M27.21b : attendu une liste de 2 : le plus grand écart de Q (un nombre), puis le booléen.")
    else:
        check_rounding("M27.21b", "le plus grand écart de Q sur 200 pas alternés", answer_M27_21b[0], 1e-12, WRONG_C21B)
        check_answer("M27.21b", answer_M27_21b[1])

with checked("M27.21"):
    if not filled(answer_M27_21c):
        print("⏳ Ex M27.21c : pas encore fait.")
    elif not isinstance(answer_M27_21c, (list, tuple)) or len(answer_M27_21c) != 2:
        print("❌ Ex M27.21c : attendu une liste de 2 : le booléen, puis le plus grand écart (un nombre).")
    else:
        check_answer("M27.21c", answer_M27_21c[0])
        check_rounding("M27.21c", "le plus grand écart de ‖x‖² + ‖y‖² − η xᵀAy sur 100 pas alternés", answer_M27_21c[1],
                       1e-12, WRONG_C21C)

### Ex M27.22 — W₂ de deux gaussiennes en dimension 1, à partir des moyennes et des écarts-types · `w2_gaussians_1d` 🔨 ★ ⏱️ 10 min
**Objectif :** programmer W₂ de deux gaussiennes de la droite, avec des écarts-types et sans élever au carré, et le lire comme une distance euclidienne.  
**Prérequis :** fiche §8 ; Ex M27.8 et M27.16 · **mymath :** `distances.py` · **Parcours :** C, E

> Même mode d'emploi qu'en M27.19 : écris la fonction dans ta copie `mon_travail/mymath/distances.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `w2_gaussians_1d(mu1, sigma1, mu2, sigma2)` (fiche §8), en suivant sa docstring : $W_2$ entre $\mathcal{N}(\mu_1, \sigma_1^2)$ et $\mathcal{N}(\mu_2, \sigma_2^2)$ à partir des **écarts-types** ; contrôle les quatre nombres (des réels finis, pas des booléens) et le signe des écarts-types (un écart-type nul est permis : la loi est alors une masse ponctuelle) ; rends $W_2$, pas son carré, en `float` Python (`math.hypot`).

**L'usage.** Avec ta fonction :
- a) les sachets de M27.8 (`BAGS_BEFORE`, `BAGS_AFTER`, des couples (moyenne, écart-type)) (un nombre), à comparer à ta réponse c) ;
- b) la machine du 🔗 M27.16 (`SHAFTS_BEFORE`, `SHAFTS_AFTER`) (un nombre), à comparer à ta réponse b) ;
- c) $\mathcal{N}(10 ;\ 3^2)$ contre $\mathcal{N}(10 ;\ 1^2)$ (un nombre).

Ce qu'il faut observer : $W_2$ est la distance euclidienne entre les points (moyenne, écart-type) du plan.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M27")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M27, "test_w2_gaussians_1d_")

**L'usage** : les sachets, la machine, et deux gaussiennes de même moyenne.

In [ ]:
answer_M27_22a = answer_M27_22b = answer_M27_22c = ...
with wbm.attempt("M27.22"):          # if you call w2_gaussians_1d here: ⏳ while it is not written
    answer_M27_22a = ...  # a) a number, as returned: the bags, BAGS_BEFORE against BAGS_AFTER
    answer_M27_22b = ...  # b) a number, as returned: the machine, SHAFTS_BEFORE against SHAFTS_AFTER
    answer_M27_22c = ...  # c) a number, as returned: N(10, 3²) against N(10, 1²)

In [ ]:
for letter, answer in zip("abc", [answer_M27_22a, answer_M27_22b, answer_M27_22c]):
    check_answer(f"M27.22{letter}", answer)

### Ex M27.23 — `scipy.optimize.linprog` : le transport optimal entre deux lois discrètes de la droite comme programme linéaire, comparé au plan monotone et à l'aire entre les fonctions de répartition 📦 ★★ ⏱️ 25 min
**Objectif :** écrire le transport optimal comme un programme linéaire, le résoudre par une bibliothèque, et voir que sur la droite le plan monotone suffit, pas dans le plan.  
**Prérequis :** fiche §7 et §8 ; M10 ; Ex M27.7 · **Parcours :** C

Écris `transport_lp_23(C, a, b)`, qui résout le programme linéaire du transport (fiche §7) par `scipy.optimize.linprog(..., method="highs")` : les $mn$ inconnues $\gamma_{ij}$ rangées ligne par ligne ($\gamma_{ij}$ est l'inconnue $i\,n + j$), le coût $\sum\gamma_{ij}c_{ij}$, les $m + n$ contraintes d'égalité des marges (l'une est redondante : `linprog` s'en accommode) et $\gamma_{ij} \ge 0$ ; elle rend `(plan, cost)`, le plan de forme $(m, n)$ et son coût. Écris aussi `northwest_23(a, b)`, qui rend le plan du coin nord-ouest (fiche §8) pour deux tableaux de masses de même somme, les points étant supposés rangés dans l'ordre croissant.

Avec les dépôts et les magasins de M27.7 (la cellule de données de la partie A : `DEPOTS_KM`, `DEPOTS_T`, `STORES_KM`, `STORES_T`, masses en tonnes) :
- a) pour le coût $c_{ij} = \lvert x_i - y_j\rvert$ : le coût optimal (t·km), $W_1$ (km), et vrai ou faux, le plan du coin nord-ouest a le même coût, à $10^{-9}$ près (une liste de 3 : les deux nombres, puis le booléen) ;
- b) pour le coût $(x_i - y_j)^2$ : le coût optimal (t·km²), $W_2$ (km), et vrai ou faux, le plan de `linprog` est le plan du coin nord-ouest, à $10^{-6}$ près (une liste de 3 : les deux nombres, puis le booléen) ;
- c) on quitte la route : les dépôts sont aux points $(1, 0)$, $(4, 3)$ et $(6, -2)$ d'une carte, les magasins aux points $(2, -3)$, $(3, 4)$ et $(8, 1)$ (en km ; `DEPOTS_MAP`, `STORES_MAP`), avec les mêmes masses et le coût $\lVert\mathbf{x}_i - \mathbf{y}_j\rVert$ : le coût optimal, puis le coût du plan du coin nord-ouest pris dans l'ordre des abscisses (une liste de 2 nombres, t·km ; 2 décimales dans ta copie).

Compare enfin a) à ton $W_1$ de M27.7, obtenu aussi par l'aire entre les fonctions de répartition. La vérification essaie aussi `northwest_23` et `transport_lp_23` sur des cas que tu ne vois pas. Ce qu'il faut observer : sur la droite, le plan monotone est optimal pour les deux coûts, et l'aire entre les fonctions de répartition donne $W_1$ sans programme linéaire ; pour le coût $\lvert x - y\rvert$, l'optimum n'est pas toujours unique, et `linprog` peut rendre un autre plan de même coût ; dans le plan, l'ordre ne veut plus rien dire, et il faut le programme linéaire.

In [ ]:
DEPOTS_MAP = np.array([[1.0, 0.0], [4.0, 3.0], [6.0, -2.0]])    # c): the depots on a map (km)
STORES_MAP = np.array([[2.0, -3.0], [3.0, 4.0], [8.0, 1.0]])    # the stores on the map (km)

In [ ]:
def transport_lp_23(C, a, b):
    """The optimal transport plan by scipy.optimize.linprog(..., method="highs"): the m n unknowns γ_ij row by row (γ_ij is
    the unknown i n + j), the cost Σ γ_ij c_ij, the m + n equalities of the margins, γ_ij >= 0. Returns (plan, cost): the
    (m, n) plan and its cost."""
    raise NotImplementedError("transport_lp_23 n'est pas encore écrite (04_indices.md si tu bloques)")


def northwest_23(a, b):
    """The plan of the north-west corner rule for two arrays of masses of the same sum (the points in increasing order):
    from the top left cell, the largest mass possible, then down if the row is empty, right if the column is full."""
    raise NotImplementedError("northwest_23 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M27_23a = answer_M27_23b = answer_M27_23c = ...
with wbm.attempt("M27.23"):          # your two functions: ⏳ while they are not written
    answer_M27_23a = ...  # a) a list of 3: the optimal cost for |x − y| (t·km), W1 (km), then True or False: the north-west plan has the same cost (to 1e-9)
    answer_M27_23b = ...  # b) a list of 3: the optimal cost for (x − y)² (t·km²), W2 (km), then True or False: the plan of linprog is the north-west plan (to 1e-6)
    answer_M27_23c = ...  # c) a list of 2 numbers, as computed: on the map, the optimal cost, then the cost of the north-west plan in the order of the abscissas (t·km)

In [ ]:
for letter, answer in zip("abc", [answer_M27_23a, answer_M27_23b, answer_M27_23c]):
    check_answer(f"M27.23{letter}", answer)

NW_C23 = [{'a': [3.0, 5.0, 2.0, 6.0],
  'b': [4.0, 4.0, 8.0],
  'want': [[3.0, 0.0, 0.0], [1.0, 4.0, 0.0], [0.0, 0.0, 2.0], [0.0, 0.0, 6.0]]},
 {'a': [7.0, 1.0, 4.0], 'b': [5.0, 2.0, 5.0], 'want': [[5.0, 2.0, 0.0], [0.0, 0.0, 1.0], [0.0, 0.0, 4.0]]}]   # masses you do not see, and the north-west plans expected
LP_C23 = {'C': [[3.2, 7.4, 7.9, 3.7], [0.8, 1.7, 2.8, 0.8], [4.5, 4.4, 1.8, 0.8]],
 'a': [0.15, 0.6, 0.25],
 'b': [0.35, 0.1, 0.3, 0.25],
 'optimum': 1.6}   # a cost matrix you do not see, its masses, and the optimal cost

with checked("M27.23"):
    results_23 = []
    report_23 = reporter(results_23)
    for case_c23 in NW_C23:
        a_c23, b_c23 = np.array(case_c23["a"]), np.array(case_c23["b"])
        got_c23 = northwest_23(a_c23.copy(), b_c23.copy())
        if matches(got_c23, case_c23["want"], rtol=1e-9, atol=1e-9):
            continue
        arr_c23 = as_floats(got_c23)
        if arr_c23 is not None and arr_c23.shape == (len(b_c23), len(a_c23)):
            why_c23 = "ton plan a la forme (n, m) : une ligne par masse de a (les départs), une colonne par masse de b"
        elif arr_c23 is not None and arr_c23.shape == (len(a_c23), len(b_c23)) and not np.allclose(arr_c23.sum(axis=1), a_c23):
            why_c23 = ("les sommes des lignes doivent redonner a, celles des colonnes b : descends quand la ligne est "
                       "vide, va à droite quand la colonne est pleine")
        else:
            why_c23 = f"attendu {shown(case_c23['want'])}, reçu {shown(got_c23)[:160]}"
        report_23(False, "", f"northwest_23, des masses que tu ne vois pas (a = {case_c23['a']}, b = {case_c23['b']}) : "
                  f"{why_c23}")
    report_23(True, "northwest_23 : des masses que tu ne vois pas, dont une ligne et une colonne qui se vident ensemble", "")
    C_c23, a_c23, b_c23 = (np.array(LP_C23[key]) for key in ("C", "a", "b"))
    plan_c23, cost_c23 = pair("M27.23", "transport_lp_23", transport_lp_23(C_c23.copy(), a_c23.copy(), b_c23.copy()),
                              names=("plan", "cost"))
    plan_c23 = np.asarray(plan_c23, dtype=float)
    report_23(plan_c23.shape == C_c23.shape, "transport_lp_23 rend un plan de la forme de C",
              f"transport_lp_23 : le plan doit avoir la forme de C, {C_c23.shape}, reçu {plan_c23.shape}")
    report_23(np.allclose(plan_c23.sum(axis=1), a_c23, atol=1e-7) and np.allclose(plan_c23.sum(axis=0), b_c23, atol=1e-7)
              and plan_c23.min() >= -1e-9, "son plan respecte les deux marges et reste positif",
              "transport_lp_23 : le plan doit avoir pour sommes des lignes a, pour sommes des colonnes b, et des "
              "coefficients ≥ 0 (les inconnues sont rangées ligne par ligne : γ_ij est l'inconnue i n + j)")
    report_23(abs(float(cost_c23) - float((plan_c23 * C_c23).sum())) <= 1e-7,
              "le coût rendu est celui du plan rendu", "transport_lp_23 : cost doit être le coût du plan rendu, Σ γᵢⱼ cᵢⱼ")
    report_23(abs(float(cost_c23) - LP_C23["optimum"]) <= 1e-7,
              f"une matrice de coûts que tu ne vois pas (3 × 4) : le coût optimal, {fr(LP_C23['optimum'], 4)}",
              f"une matrice de coûts que tu ne vois pas : attendu le coût optimal {fr(LP_C23['optimum'], 4)}, reçu "
              f"{fr(float(cost_c23), 4)} (le programme minimise Σ γᵢⱼ cᵢⱼ)")
    verdict("M27.23", all(results_23), "tes deux fonctions passent les cas cachés.", "")

**Après la vérification**, dans ta copie : pourquoi le plan du coin nord-ouest est-il optimal sur la route, pour les deux coûts ? Pourquoi `linprog` peut-il rendre un autre plan pour $\lvert x - y\rvert$, et pas pour $(x - y)^2$ ? Que perd-on sur la carte en rangeant les points par abscisse ?

## Partie B · Prédire, mesurer, dessiner

Fiche §4, §8, §9 et §10. La trajectoire de la descente-montée, prédite puis tracée ; le biais d'une distance estimée, simulé ; deux fonctions de répartition, l'aire entre elles et leur plus grand écart, dessinés.

### Ex M27.24 — Prédire la trajectoire de la descente-montée simultanée puis alternée sur xy, puis la tracer : spirale qui s'éloigne ou courbe fermée ? 🔮 ★ ⏱️ 15 min
**Objectif :** prévoir par le module et l'argument des valeurs propres d'un pas la trajectoire de la descente-montée, puis la tracer, et voir le pas alterné quitter sa courbe fermée pour η > 2.  
**Prérequis :** fiche §4 ; M22 ; Ex M27.4 et M27.21 · **Parcours :** C

On joue la descente-montée sur $f(x, y) = xy$ depuis $(x_0, y_0) = (1, 0)$, pendant 40 pas. **Avant d'exécuter quoi que ce soit**, prédis :
- a) avec $\eta = 0{,}3$, en simultané, la distance à l'équilibre après 40 pas vaut environ : (A) 1 ; (B) 1,5 ; (C) 5,6 ; (D) 40 (une lettre) ;
- b) avec $\eta = 0{,}3$, en simultané, le nombre de tours que font les itérés autour de l'équilibre en 40 pas : (A) moins d'un ; (B) entre 1 et 2 ; (C) entre 3 et 4 ; (D) aucun, ils partent en ligne droite (une lettre) ;
- c) avec $\eta = 2{,}5$, en alterné : (A) une courbe fermée ; (B) des itérés qui s'éloignent ; (C) des itérés qui convergent vers $(0, 0)$ (une lettre).

Puis, une fois tes prédictions écrites et vérifiées, écris `draw_paths_24(ax, eta, n_steps, alternating)`, qui dessine sur l'**axe reçu** (sans ouvrir de figure), avec ta `descent_ascent_iterates` (M27.21), les itérés depuis $(1, 0)$ reliés dans l'ordre ($x$ en abscisse, $y$ en ordonnée), l'équilibre marqué par un point, des axes de même échelle (`ax.set_aspect("equal")`), un titre qui donne $\eta$ et le mode (« simultané » ou « alterné »), et des noms d'axes. La vérification l'appelle pour $\eta = 0{,}3$ (simultané, puis alterné) et pour $\eta = 2{,}5$ (alterné), et affiche la figure ; elle l'appelle aussi avec un autre pas, que tu ne vois pas. Relève :
- d) la distance à l'équilibre après 40 pas simultanés avec $\eta = 0{,}3$ (un nombre ; 4 décimales dans ta copie) ;
- e) le nombre de tours de ces 40 pas : l'angle total parcouru, `np.unwrap(np.arctan2(y, x))` du premier au dernier itéré, divisé par $2\pi$ (un nombre ; 4 décimales dans ta copie) ;
- f) $\log_{10}$ de la distance à l'équilibre après 40 pas alternés avec $\eta = 2{,}5$ (un nombre ; 2 décimales dans ta copie).

La vérification de l'expérience ne s'exécute qu'une fois tes trois prédictions écrites. Ce qu'il faut observer : la spirale est réglée par le module et l'argument de $1 \pm i\eta$ ; le pas alterné ne reste sur une courbe fermée que pour $\eta < 2$.

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M27_24a = ...  # a) "A", "B", "C" or "D", written BEFORE the experiment: η = 0.3, simultaneous, the distance after 40 steps is about...
prediction_M27_24b = ...  # b) "A", "B", "C" or "D": the number of turns of these 40 steps
prediction_M27_24c = ...  # c) "A", "B" or "C": η = 2.5, alternating

In [ ]:
for letter, prediction in zip("abc", [prediction_M27_24a, prediction_M27_24b, prediction_M27_24c]):
    wbm.check(f"M27.24{letter}", prediction)

**L'expérience.** Écris-la une fois tes trois prédictions vérifiées ; sa vérification ne s'exécute qu'à cette condition.

In [ ]:
def draw_paths_24(ax, eta, n_steps, alternating):
    """In ax, with your descent_ascent_iterates: the iterates on xy from (1, 0), joined in order (x horizontal, y
    vertical); the equilibrium (0, 0) marked by a point; axes of the same scale (ax.set_aspect("equal")); a title that
    gives eta and the mode (« simultané » or « alterné »); axis labels. No other axes."""
    raise NotImplementedError("draw_paths_24 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M27_24d = answer_M27_24e = answer_M27_24f = ...
with wbm.attempt("M27.24"):          # your descent_ascent_iterates: ⏳ while it is not written
    answer_M27_24d = ...  # d) a number, as computed: the distance to (0, 0) after 40 simultaneous steps, η = 0.3
    answer_M27_24e = ...  # e) a number, as computed: the number of turns of these 40 steps (unwrapped angle / 2π)
    answer_M27_24f = ...  # f) a number, as computed: log10 of the distance after 40 alternating steps, η = 2.5

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M27_24a, prediction_M27_24b, prediction_M27_24c]):
    print("⏳ Écris d'abord tes trois prédictions, puis exécute à nouveau.")
else:
    PATHS_C24 = {'0.3-s': [[1.0, 1.0, 0.91, 0.73, 0.4681, 0.1405, -0.22922900000000002, -0.611603, -0.9733463900000001,
            -1.28004551, -1.4991434549000002, -1.6030373039000003, -1.5720082419590005, -1.3967058226670006,
            -1.0799226615986905, -0.6374359764903506, -0.09775625183812842, 0.49929271069822534,
            1.1051397359000106, 1.6660504171389556, 2.1274985221468996, 2.4390020896123374, 2.5590307900845546,
            2.459549302491661, 2.1297550437911577, 1.578601347866405, 0.835769698000448, -0.04913607317348534,
            -1.0092611171674588, -1.9649639145758186, -2.829833211439107, -3.517855755990572,
            -3.951193311512517, -4.067923848995311, -3.8290469884419784, -3.224056981479068, -2.274452745556379,
            -1.0346833813005742, 0.40978673005530464, 1.947378345728235, 3.448089155696188],
           [0.0, 0.3, 0.6, 0.873, 1.092, 1.2324300000000001, 1.27458, 1.2058113000000001, 1.0223304000000002,
            0.7303264830000002, 0.34631283000000024, -0.10343020646999984, -0.5843413976399999,
            -1.0559438702277, -1.4749556170278002, -1.7989324155074073, -1.9901632084545124, -2.019490084005951,
            -1.8697022707964832, -1.53816035002648, -1.0383452248847933, -0.4000956682407234,
            0.3316049586429778, 1.099314195668344, 1.8371789864158425, 2.47610549955319, 2.949685903913111,
            3.2004168133132453, 3.1856759913611996, 2.882897656210962, 2.2934084818382163, 1.4444585184064844,
            0.3891017916093129, -0.7962562018444421, -2.0166333565430357, -3.165347453075629,
            -4.132564547519349, -4.814900371186263, -5.125305385576435, -5.002369366559844,
            -4.418155862841373]],
 '0.3-a': [[1.0, 1.0, 0.91, 0.7381000000000001, 0.4997710000000001, 0.2164626100000001, -0.0863274148999999,
            -0.38134797245899993, -0.64204721249669, -0.8449622034096779, -0.9718305960157948,
            -1.0112342349804901, -0.9596267927969414, -0.8216529392616679, -0.6097303211928444,
            -0.3429319742166649, -0.045269749560985506, 0.25646675255518253, 0.5351212469413842,
            0.7656148291028612, 0.9272030766450806, 1.0053430472892428, 0.9930021436773733, 0.8912910471345401,
            0.7093637563495984, 0.4635937274931927, 0.17610026316239974, -0.12724222485300918,
            -0.41913291263164726, -0.6733016382734371, -0.8668732164706177, -0.9824262051854427,
            -1.0095608354335779, -0.945834990492691, -0.796983996407462, -0.5764044426455613,
            -0.3039484890455602, -0.004137171431458642, 0.29604649161147417, 0.5695859704093743,
            0.7918627118704308],
           [0.0, 0.3, 0.573, 0.79443, 0.9443613, 1.009300083, 0.9834018585300001, 0.8689974667923,
            0.6763833030432931, 0.42289464202038973, 0.13134546321565133, -0.17202480727849573,
            -0.45991284511757813, -0.7064087268960785, -0.8893278232539318, -0.9922074155189312,
            -1.0057883403872268, -0.928848314620672, -0.7683119405382567, -0.5386274918073983,
            -0.26046656881387414, 0.041136345372898675, 0.33903698847611063, 0.6064243026164726,
            0.8192334295213521, 0.9583115477693099, 1.0111416267180298, 0.972968959262127, 0.8472290854726329,
            0.6452385939906018, 0.3851766290494165, 0.09044876749378372, -0.2124194831362896,
            -0.4961699802840969, -0.7352651792063355, -0.9081865120000039, -0.9993710587136719,
            -1.0006122101431094, -0.9117982626596671, -0.7409224715368549, -0.5033636579757257]],
 '2.5-a': [[1.0, 1.0, -5.25, 21.3125, -85.328125, 341.33203125, -1365.3330078125, 5461.333251953125,
            -21845.33331298828, 87381.33332824707, -349525.33333206177, 1398101.3333330154, -5592405.333333254,
            22369621.333333313, -89478485.33333333, 357913941.3333333, -1431655765.3333333, 5726623061.333333,
            -22906492245.333332, 91625968981.33333, -366503875925.3333, 1466015503701.3333, -5864062014805.333,
            23456248059221.332, -93824992236885.33, 375299968947541.3, -1501199875790165.2, 6004799503160661.0,
            -2.4019198012642644e+16, 9.607679205057058e+16, -3.843071682022823e+17, 1.5372286728091292e+18,
            -6.148914691236517e+18, 2.4595658764946067e+19, -9.838263505978427e+19, 3.935305402391371e+20,
            -1.5741221609565483e+21, 6.296488643826193e+21, -2.5185954575304773e+22, 1.007438183012191e+23,
            -4.029752732048764e+23],
           [0.0, 2.5, -10.625, 42.65625, -170.6640625, 682.666015625, -2730.66650390625, 10922.666625976562,
            -43690.66665649414, 174762.66666412354, -699050.6666660309, 2796202.6666665077, -11184810.666666627,
            44739242.66666666, -178956970.66666666, 715827882.6666666, -2863311530.6666665, 11453246122.666666,
            -45812984490.666664, 183251937962.66666, -733007751850.6666, 2932031007402.6665,
            -11728124029610.666, 46912496118442.664, -187649984473770.66, 750599937895082.6,
            -3002399751580330.5, 1.2009599006321322e+16, -4.803839602528529e+16, 1.9215358410114115e+17,
            -7.686143364045646e+17, 3.0744573456182584e+18, -1.2297829382473034e+19, 4.9191317529892135e+19,
            -1.9676527011956854e+20, 7.870610804782742e+20, -3.1482443219130966e+21, 1.2592977287652387e+22,
            -5.037190915060955e+22, 2.014876366024382e+23, -8.059505464097527e+23]],
 'hidden': [[1.0, 1.0, 0.7975, 0.43350625, -0.018272515625000096, -0.4663510968359376, -0.8199935809375978,
             -1.0075873648993943, -0.9911447074690635, -0.7739952467762474, -0.4001117486112412,
             0.054794378647541386, 0.49860464423019685],
            [0.0, 0.45, 0.808875, 1.0039528125000001, 0.9957301804687501, 0.7858721868925781,
             0.4168750754706591, -0.036539238734068336, -0.4825543570951469, -0.8308522181444582,
             -1.0109025050195168, -0.9862450346281232, -0.7618729447245346]]}   # the iterates expected, [xs, ys], for each call of the check


    def look_24(ax, eta, n_steps, alternating, key):
        """What is missing in a drawing of draw_paths_24 (French phrases)."""
        xs, ys = PATHS_C24[key]
        missing = []
        if not through(ax, xs, ys, rtol=1e-9, atol=1e-12):
            missing.append(f"les {n_steps + 1} itérés depuis (1, 0), reliés dans l'ordre (x en abscisse, y en ordonnée)")
        if not points_at(ax, [0.0], [0.0], marker_only=True):
            missing.append("l'équilibre (0, 0), marqué par un point")
        if ax.get_aspect() != 1.0:
            missing.append('des axes de même échelle (ax.set_aspect("equal"))')
        title = ax.get_title().lower()
        eta_texts = (f"{eta:g}", f"{eta:g}".replace(".", ","))
        if not any(text in title for text in eta_texts) or ("altern" if alternating else "simultan") not in title:
            missing.append(f"un titre qui donne η ({eta_texts[1]}) et le mode ({'alterné' if alternating else 'simultané'})")
        if not ax.get_xlabel() or not ax.get_ylabel():
            missing.append("des noms d'axes")
        return missing


    with checked("M27.24"):
        results_24 = []
        report_24 = reporter(results_24, stop=False)
        fig_c24, ax_c24 = plt.subplots()
        try:
            with own_code("M27.24", "draw_paths_24", REF_DISTANCES):
                opened_c24 = drawn(draw_paths_24, ax_c24, 0.45, 12, True)
            missing_c24 = look_24(ax_c24, 0.45, 12, True, "hidden")
        finally:
            plt.close(fig_c24)
        fig_24, axes_24 = plt.subplots(1, 3, figsize=(15, 4.6))
        try:
            opened_24 = list(opened_c24)
            for ax_24, (eta_24, alternating_24, key_24) in zip(axes_24, ((0.3, False, "0.3-s"), (0.3, True, "0.3-a"),
                                                                         (2.5, True, "2.5-a"))):
                with own_code("M27.24", "draw_paths_24", REF_DISTANCES):
                    opened_24 += drawn(draw_paths_24, ax_24, eta_24, 40, alternating_24)
                missing_24 = look_24(ax_24, eta_24, 40, alternating_24, key_24)
                mode_24 = "alterné" if alternating_24 else "simultané"
                report_24(not missing_24, f"η = {fr(eta_24, 1)}, {mode_24} : les 41 itérés, l'équilibre, l'échelle, le titre et "
                          "les axes sont là", f"η = {fr(eta_24, 1)}, {mode_24} : il manque " + " ; ".join(missing_24))
            report_24(not missing_c24, "un appel que tu ne vois pas (un autre pas, 12 pas alternés) est dessiné juste aussi",
                      "un appel que tu ne vois pas (un autre pas, 12 pas alternés) : il manque " + " ; ".join(missing_c24))
            report_24(not opened_24, "draw_paths_24 dessine dans l'axe reçu",
                      "draw_paths_24 doit dessiner dans l'axe reçu, sans ouvrir de figure")
        except BaseException:
            plt.close(fig_24)
            raise
        fig_24.tight_layout()
        plt.show()
        verdict("M27.24", all(results_24), "tes trois trajectoires sont dessinées.", "relis les lignes ❌ ci-dessus.")

    for letter, answer in zip("def", [answer_M27_24d, answer_M27_24e, answer_M27_24f]):
        check_answer(f"M27.24{letter}", answer)

**Après l'expérience**, dans ta copie : pourquoi la spirale fait-elle presque deux tours en 40 pas ? Que deviennent la spirale et l'ellipse avec $\eta = 0{,}1$ ? Pourquoi le pas alterné saute-t-il d'un côté à l'autre de l'équilibre pour $\eta = 2{,}5$ ?

### Ex M27.25 — Le biais d'une distance estimée : ‖x̄₁ − x̄₂‖² entre deux échantillons d'une même loi de ℝ³, selon leurs tailles, contre tr Σ₁/n₁ + tr Σ₂/n₂ 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer par simulation le biais d'une distance estimée entre deux moyennes, le comparer à tr Σ (1/n₁ + 1/n₂), et voir un petit décalage réel s'y noyer.  
**Prérequis :** fiche §9 ; M02c ; Ex M27.15 · **Parcours :** C

Une machine fabrique des cales ; on mesure trois cotes de chacune (en mm), de moyenne $\boldsymbol{\mu} = (80, 25, 6)$ (`MU_25`) et de covariance
$$\boldsymbol{\Sigma} = \begin{pmatrix} 0{,}09 & 0{,}03 & 0 \\ 0{,}03 & 0{,}04 & 0{,}01 \\ 0 & 0{,}01 & 0{,}01 \end{pmatrix}, \qquad \operatorname{tr}\boldsymbol{\Sigma} = 0{,}14\ \text{mm}^2$$
(`SIGMA_25`). Écris `distance_bias_25(mu1, mu2, Sigma, n1, n2, n_rep, rng)`, qui simule `n_rep` paires de séries indépendantes, toutes à la fois avec des tableaux, dans cet ordre : `L = np.linalg.cholesky(Sigma)`, puis la série 1, `mu1 + rng.standard_normal((n_rep, n1, 3)) @ L.T`, puis la série 2, `mu2 + rng.standard_normal((n_rep, n2, 3)) @ L.T` ; elle rend la moyenne, sur les `n_rep` paires, de $\lVert\bar{\mathbf{x}}_1 - \bar{\mathbf{x}}_2\rVert^2$. Écris aussi `draw_bias_25(ax, ns, means, trace)`, qui trace dans l'axe reçu, en échelles logarithmiques, les moyennes des séries de même taille $n$ contre $n$ (des points) et la courbe $2\,\text{trace}/n$ sur toute l'étendue des `ns`, avec des noms d'axes et une légende.

Avec `rng = np.random.default_rng(6425)`, `n_rep = 4000` (`N_REP_25`) et ce même `rng` pour tous les appels, dans l'ordre : deux séries de la même machine (`mu1 = mu2 = MU_25`) pour $(n_1, n_2) = (5, 5)$, $(20, 20)$, $(80, 80)$, $(5, 80)$ et $(80, 5)$ (`SIZES_25`) ; puis, avec `mu2 = MU_25 + SHIFT_25` (un vrai décalage $\boldsymbol{\delta} = (0{,}05 ;\ 0 ;\ 0)$ de la première cote), pour $(5, 5)$ et $(80, 80)$. Relève :
- a) les cinq moyennes, les cinq valeurs $\operatorname{tr}\boldsymbol{\Sigma}\,(\frac{1}{n_1} + \frac{1}{n_2})$, et leurs rapports, moyenne divisée par valeur (une liste de 3 listes de 5 nombres ; 3 décimales pour les rapports dans ta copie) ;
- b) vrai ou faux : les cinq rapports sont à moins de 5 % de 1 ;
- c) avec le décalage, les deux moyennes, puis les mêmes moins $\operatorname{tr}\boldsymbol{\Sigma}\,(\frac{1}{n_1} + \frac{1}{n_2})$, à comparer à $\lVert\boldsymbol{\delta}\rVert^2 = 0{,}0025$ (une liste de 4 nombres) ;
- d) vrai ou faux : avec 5 cales par série, la distance estimée moyenne dépasse vingt fois la vraie distance $\lVert\boldsymbol{\delta}\rVert^2$.

La vérification essaie `distance_bias_25` sur un cas que tu ne vois pas, puis appelle `draw_bias_25` sur les trois tailles égales (avec les moyennes attendues, arrondies) et affiche la figure. Ce qu'il faut conclure : la distance estimée entre deux séries d'une même machine n'est pas nulle, elle suit $\operatorname{tr}\boldsymbol{\Sigma}\,(\frac{1}{n_1} + \frac{1}{n_2})$ ; un petit décalage réel se noie dans ce biais sur de petites séries ; deux distances estimées ne se comparent qu'à tailles égales, ou après avoir retiré le biais.

In [ ]:
MU_25 = np.array([80.0, 25.0, 6.0])                       # the mean of the three dimensions of a shim (mm)
SIGMA_25 = np.array([[0.09, 0.03, 0.0], [0.03, 0.04, 0.01], [0.0, 0.01, 0.01]])   # their covariance (mm²), tr = 0.14
SIZES_25 = [(5, 5), (20, 20), (80, 80), (5, 80), (80, 5)]  # (n1, n2): two series of the same machine
SHIFT_25 = np.array([0.05, 0.0, 0.0])                     # a true shift of the first dimension (mm)
N_REP_25 = 4000                                           # the number of simulated pairs of series

In [ ]:
def distance_bias_25(mu1, mu2, Sigma, n1, n2, n_rep, rng):
    """n_rep pairs of independent series at once: L = np.linalg.cholesky(Sigma), then the series 1,
    mu1 + rng.standard_normal((n_rep, n1, 3)) @ L.T, then the series 2, mu2 + rng.standard_normal((n_rep, n2, 3)) @ L.T;
    returns the mean over the n_rep pairs of ‖x̄1 − x̄2‖²."""
    raise NotImplementedError("distance_bias_25 n'est pas encore écrite (04_indices.md si tu bloques)")


def draw_bias_25(ax, ns, means, trace):
    """In ax, in log scales on both axes: the means against n (points), and the curve 2 trace / n over the whole range of
    ns; axis labels and a legend. No other axes."""
    raise NotImplementedError("draw_bias_25 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M27_25a = answer_M27_25b = answer_M27_25c = answer_M27_25d = ...
with wbm.attempt("M27.25"):          # your distance_bias_25: ⏳ while it is not written
    answer_M27_25a = ...  # a) a list of 3 lists of 5 numbers, as computed: the five means, the five values tr Σ (1/n1 + 1/n2), their ratios (mean / value)
    answer_M27_25b = ...  # b) True or False: the five ratios are within 5 % of 1
    answer_M27_25c = ...  # c) a list of 4 numbers, as computed: with the shift, the means for (5, 5) and (80, 80), then the same minus tr Σ (1/n1 + 1/n2)
    answer_M27_25d = ...  # d) True or False: with 5 shims per series, the mean estimated distance exceeds twenty times ‖δ‖²

In [ ]:
MEANS_C25 = [0.056, 0.0138, 0.0036]   # the means of a) for (5, 5), (20, 20), (80, 80), rounded to 4 decimals (the figure, a hint)


def means_hint_25(value):
    """a): means close to the expected ones (within 10 %), not equal to them: the order of the draws."""
    got = np.asarray(value, dtype=float)
    if got.shape == (3, 5) and np.all(np.abs(got[0, :3] / np.array(MEANS_C25) - 1) < 0.1):
        return ("tes moyennes ont le bon ordre de grandeur : les valeurs exactes demandent les tirages dans l'ordre de "
                "l'énoncé (à chaque appel, la série 1 puis la série 2 ; les appels dans l'ordre de SIZES_25), avec le "
                "même rng")
    return None


check_answer("M27.25a", answer_M27_25a, hint=means_hint_25)
check_answer("M27.25b", answer_M27_25b)
check_answer("M27.25c", answer_M27_25c)
check_answer("M27.25d", answer_M27_25d)

HIDDEN_C25 = {'mu1': [1.0, 2.0, 3.0],
 'mu2': [1.1, 2.0, 2.9],
 'Sigma': [[0.5, 0.1, 0.0], [0.1, 0.3, 0.0], [0.0, 0.0, 0.2]],
 'n1': 7,
 'n2': 3,
 'n_rep': 500,
 'seed': 6426,
 'want': 0.5119650377703837,
 'second_first': 0.5156903453516299,
 'no_cholesky': 0.21531199682623123,
 'norm': 0.6561501697180852}   # a case you do not see: its data, the value expected, three classic wrong values

with checked("M27.25"):
    results_25 = []
    report_25 = reporter(results_25, stop=False)
    h_c25 = HIDDEN_C25
    got_c25 = distance_bias_25(np.array(h_c25["mu1"]), np.array(h_c25["mu2"]), np.array(h_c25["Sigma"]), h_c25["n1"],
                               h_c25["n2"], h_c25["n_rep"], np.random.default_rng(h_c25["seed"]))
    try:
        value_c25 = float(got_c25)
    except (TypeError, ValueError):
        value_c25 = float("nan")
    why_c25 = next((text for key, text in (
        ("second_first", "tu tires la série 2 avant la série 1 : la série 1 d'abord, avec le même rng"),
        ("no_cholesky", "les tirages passent par L = np.linalg.cholesky(Sigma) : x = mu + z @ L.T a pour covariance "
                        "L Lᵀ = Σ"),
        ("norm", "la moyenne porte sur ‖x̄₁ − x̄₂‖², le carré de la norme")) if math.isclose(value_c25, h_c25[key],
                                                                                          rel_tol=1e-9)),
                   f"attendu {h_c25['want']:.6g}, reçu {shown(got_c25)}")
    report_25(math.isclose(value_c25, h_c25["want"], rel_tol=1e-9),
              "distance_bias_25 : un cas que tu ne vois pas (deux moyennes différentes, n1 = 7, n2 = 3), comme attendu",
              f"distance_bias_25, un cas que tu ne vois pas (n1 = 7, n2 = 3, default_rng({h_c25['seed']})) : {why_c25}")
    fig_c25, ax_c25 = plt.subplots()
    try:
        opened_c25 = drawn(draw_bias_25, ax_c25, [3, 9, 27, 81], [0.4, 0.12, 0.035, 0.013], 0.5)
        hidden_ok_c25 = (points_at(ax_c25, [3, 9, 27, 81], [0.4, 0.12, 0.035, 0.013], marker_only=True)
                         and follows(ax_c25, lambda n: 2 * 0.5 / n, 3, 81))
    finally:
        plt.close(fig_c25)
    fig_25, ax_25 = plt.subplots(figsize=(6.5, 4.2))
    try:
        opened_25 = drawn(draw_bias_25, ax_25, [5, 20, 80], list(MEANS_C25), 0.14)
        report_25(not opened_25 and not opened_c25, "draw_bias_25 dessine dans l'axe reçu",
                  "draw_bias_25 doit dessiner dans l'axe reçu, sans ouvrir de figure")
        report_25(points_at(ax_25, [5, 20, 80], MEANS_C25, marker_only=True),
                  "les trois moyennes des séries de même taille sont des points",
                  "il manque les moyennes contre n, en points (un marqueur : ax.plot(ns, means, \"o\"), ou ax.scatter)")
        report_25(follows(ax_25, lambda n: 2 * 0.14 / n, 5, 80), "la droite 2 tr Σ / n est tracée de n = 5 à n = 80",
                  "il manque la courbe y = 2 trace / n, tracée sur toute l'étendue des n reçus")
        report_25(ax_25.get_xscale() == "log" and ax_25.get_yscale() == "log", "les deux échelles sont logarithmiques",
                  "les deux échelles doivent être logarithmiques (ax.set_xscale(\"log\"), ax.set_yscale(\"log\"))")
        report_25(ax_25.get_legend() is not None and ax_25.get_xlabel() != "" and ax_25.get_ylabel() != "",
                  "des noms d'axes et une légende", "draw_bias_25 doit nommer les axes et mettre une légende")
        report_25(hidden_ok_c25, "un appel que tu ne vois pas (d'autres tailles, une autre trace) est dessiné juste aussi",
                  "un appel que tu ne vois pas (ns = [3, 9, 27, 81], trace = 0,5) : il manque les points ou la courbe "
                  "2 trace / n")
    except BaseException:
        plt.close(fig_25)
        raise
    fig_25.tight_layout()
    plt.show()
    verdict("M27.25", all(results_25), "ta simulation et ta figure sont justes.", "relis les lignes ❌ ci-dessus.")

**Après la figure**, dans ta copie : pourquoi la distance estimée moyenne est-elle divisée par 4 quand $n$ est multiplié par 4 ? Pourquoi $(5, 80)$ et $(80, 5)$ donnent-ils presque la même chose ? Avec 5 cales par série, pourrais-tu voir le décalage de 0,05 mm ?

### Ex M27.26 — Deux fonctions de répartition sur une même figure : l'aire entre elles (W₁) et leur plus grand écart vertical (KS), pour une translation puis une dilatation 🎨 ★ ⏱️ 20 min
**Objectif :** dessiner deux fonctions de répartition, l'aire entre elles et leur plus grand écart, et voir ce que W₁ et KS mesurent et ce qu'un changement d'unité leur fait.  
**Prérequis :** fiche §8 et §10 ; Ex M27.8 et M27.10 · **Parcours :** C, E

Écris `draw_cdf_gaps_26(ax, cdf_a, cdf_b, grid, label_a, label_b, unit)`, qui dessine sur l'**axe reçu** (sans ouvrir de figure) les deux fonctions de répartition `cdf_a` et `cdf_b` évaluées sur `grid` (une courbe chacune, de labels `label_a` et `label_b`, avec une légende), colore l'aire entre elles (`ax.fill_between`), trace un segment vertical d'une courbe à l'autre là où leur écart $\lvert F - G\rvert$ est le plus grand sur la grille, et met en titre les deux nombres $W_1$ (l'aire entre les courbes, par la méthode des trapèzes, `np.trapezoid`), avec son unité `unit`, et KS (le plus grand écart sur la grille) ; elle rend `(w1, ks)`.

La vérification l'appelle trois fois et affiche les figures : une **translation**, la température d'un local, $\mathcal{N}(20 ;\ 2^2)$ contre $\mathcal{N}(21{,}2 ;\ 2^2)$ en °C, sur `np.linspace(10, 32, 4401)` (`ROOM_A_26`, `ROOM_B_26`, `GRID_C_26`) ; une **dilatation**, une attente à un guichet, la loi exponentielle de moyenne 2 contre celle de moyenne 5, en minutes, sur `np.linspace(0, 80, 16001)` (`WAIT_A_26`, `WAIT_B_26`, `GRID_MIN_26` ; `scipy.stats.expon(scale=2).cdf`) ; la même translation en °F, $\mathcal{N}(68 ;\ 3{,}6^2)$ contre $\mathcal{N}(70{,}16 ;\ 3{,}6^2)$, sur la grille $1{,}8 \times$ `grid` $+ 32$ (`ROOM_A_F_26`, `ROOM_B_F_26`, `GRID_F_26`) ; elle l'appelle aussi sur deux autres lois, que tu ne vois pas. Relève, avec ta fonction (appelle-la sur l'axe d'une figure que tu fermes aussitôt, `fig, ax = plt.subplots()` puis `plt.close(fig)`, pour ne pas afficher la figure deux fois) :
- a) `(w1, ks)` pour la translation (une liste de 2 nombres ; 3 décimales dans ta copie) ;
- b) `(w1, ks)` pour la dilatation (une liste de 2 nombres ; 3 décimales dans ta copie) ;
- c) `(w1, ks)` en °F (une liste de 2 nombres ; 3 décimales dans ta copie).

Ce qu'il faut observer : pour la translation, l'aire vaut le décalage, et l'écart vertical est le plus grand au milieu ; pour la dilatation, les deux courbes partent du même point et s'écartent, puis se rejoignent ; un changement d'unité multiplie $W_1$ et laisse KS intact.

In [ ]:
GRID_C_26 = np.linspace(10, 32, 4401)                     # the translation: a room, in °C
ROOM_A_26, ROOM_B_26 = stats.norm(20, 2).cdf, stats.norm(21.2, 2).cdf
GRID_MIN_26 = np.linspace(0, 80, 16001)                   # the dilatation: a wait at a counter, in minutes
WAIT_A_26, WAIT_B_26 = stats.expon(scale=2).cdf, stats.expon(scale=5).cdf
GRID_F_26 = 1.8 * GRID_C_26 + 32                           # the same translation in °F
ROOM_A_F_26, ROOM_B_F_26 = stats.norm(68, 3.6).cdf, stats.norm(70.16, 3.6).cdf

In [ ]:
def draw_cdf_gaps_26(ax, cdf_a, cdf_b, grid, label_a, label_b, unit):
    """In ax: the two distribution functions on grid (labels label_a and label_b, and a legend), the area between them
    colored (ax.fill_between), a vertical segment from one curve to the other where |F − G| is largest on the grid, and a
    title with W1 (the area, np.trapezoid, with its unit) and KS (the largest gap on the grid). Returns (w1, ks). No other
    axes."""
    raise NotImplementedError("draw_cdf_gaps_26 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M27_26a = answer_M27_26b = answer_M27_26c = ...
with wbm.attempt("M27.26"):          # your function: ⏳ while it is not written
    answer_M27_26a = ...  # a) a list of 2 numbers, as computed: (w1, ks) for the translation (°C)
    answer_M27_26b = ...  # b) a list of 2 numbers, as computed: (w1, ks) for the dilatation (minutes)
    answer_M27_26c = ...  # c) a list of 2 numbers, as computed: (w1, ks) for the translation in °F

In [ ]:
for letter, answer in zip("abc", [answer_M27_26a, answer_M27_26b, answer_M27_26c]):
    check_answer(f"M27.26{letter}", answer)

CASES_C26 = [[1.199999885391605, 0.23582284437790513, 20.6], [2.999998812324181, 0.32573010606781894, 3.055],
 [2.1599997937048867, 0.23582284437790496, 69.08000000000001]]   # for each call: (w1, ks) expected, and the abscissa of the largest gap on the grid
HIDDEN_C26 = [0.666666, 0.28125, 1.25]   # a call you do not see: (w1, ks) expected and the abscissa of the largest gap


def vertical_at(ax, x, low, high, tol):
    """True when a segment of ax is vertical at x (within tol) and goes from low to high (a line or ax.vlines)."""
    segments = [np.column_stack(line_points(line)) for line in ax.lines]
    for collection in ax.collections:
        if type(collection).__name__ == "LineCollection":
            segments += [np.asarray(s, dtype=float) for s in collection.get_segments()]
    for s in segments:
        if (len(s) >= 2 and np.all(np.abs(s[:, 0] - x) <= tol) and s[:, 1].min() <= low + 1e-6
                and s[:, 1].max() >= high - 1e-6):
            return True
    return False


def look_26(ax, cdf_a, cdf_b, grid, label_a, label_b, unit, x_star):
    """What is missing in a drawing of draw_cdf_gaps_26 (French phrases)."""
    missing = []
    if not (through(ax, grid, cdf_a(grid), rtol=1e-9, atol=1e-12) and through(ax, grid, cdf_b(grid), rtol=1e-9, atol=1e-12)):
        missing.append("les deux fonctions de répartition, évaluées sur grid (une courbe chacune)")
    if not any(type(c).__name__ in ("PolyCollection", "FillBetweenPolyCollection") for c in ax.collections):
        missing.append("l'aire entre elles, colorée (ax.fill_between)")
    low, high = sorted((float(cdf_a(x_star)), float(cdf_b(x_star))))
    if not vertical_at(ax, x_star, low, high, 0.51 * (grid[1] - grid[0])):
        missing.append("le segment vertical du plus grand écart, d'une courbe à l'autre")
    title = ax.get_title()
    if "ks" not in title.lower() or "w" not in title.lower() or unit not in title:
        missing.append(f"un titre avec W₁ (et son unité, {unit}) et KS")
    legend = ax.get_legend()
    texts = [t.get_text() for t in legend.get_texts()] if legend is not None else []
    if label_a not in texts or label_b not in texts:
        missing.append("une légende avec label_a et label_b")
    return missing


def returned_26(label, result, want):
    """The (w1, ks) returned compared with want: '' when right, else the reason."""
    try:
        w1, ks = (float(v) for v in result)
    except (TypeError, ValueError):
        return f"draw_cdf_gaps_26 doit rendre le couple (w1, ks), reçu {shown(result)}"
    if math.isclose(w1, want[1], abs_tol=1e-6) and math.isclose(ks, want[0], abs_tol=1e-6):
        return "tu rends (ks, w1) : rends (w1, ks), dans cet ordre"
    if not math.isclose(w1, want[0], abs_tol=1e-6):
        return f"W₁ attendu {want[0]:.6f}, reçu {w1:.6f} (l'aire entre les deux courbes, np.trapezoid sur grid)"
    if not math.isclose(ks, want[1], abs_tol=1e-6):
        return f"KS attendu {want[1]:.6f}, reçu {ks:.6f} (le plus grand |F − G| sur grid)"
    return ""


with checked("M27.26"):
    results_26 = []
    report_26 = reporter(results_26, stop=False)
    hidden_a_c26, hidden_b_c26 = stats.uniform(0, 4).cdf, stats.triang(0.5, loc=1, scale=2).cdf
    hidden_grid_c26 = np.linspace(-1, 5, 3001)
    fig_c26, ax_c26 = plt.subplots()
    try:
        result_c26 = []
        opened_c26 = drawn(lambda: result_c26.append(draw_cdf_gaps_26(ax_c26, hidden_a_c26, hidden_b_c26,
                                                                      hidden_grid_c26.copy(), "loi A", "loi B", "m")))
        missing_c26 = look_26(ax_c26, hidden_a_c26, hidden_b_c26, hidden_grid_c26, "loi A", "loi B", "m", HIDDEN_C26[2])
        why_c26 = returned_26("caché", result_c26[0], HIDDEN_C26)
    finally:
        plt.close(fig_c26)
    calls_26 = [("une translation (°C)", ROOM_A_26, ROOM_B_26, GRID_C_26, "N(20 ; 2²)", "N(21,2 ; 2²)", "°C"),
                ("une dilatation (minutes)", WAIT_A_26, WAIT_B_26, GRID_MIN_26, "moyenne 2 min", "moyenne 5 min", "min"),
                ("la translation en °F", ROOM_A_F_26, ROOM_B_F_26, GRID_F_26, "N(68 ; 3,6²)", "N(70,16 ; 3,6²)", "°F")]
    fig_26, axes_26 = plt.subplots(1, 3, figsize=(16, 4.4))
    try:
        opened_26 = list(opened_c26)
        for ax_26, (label_26, cdf_a_26, cdf_b_26, grid_26, la_26, lb_26, unit_26), case_26 in zip(axes_26, calls_26,
                                                                                                    CASES_C26):
            result_26 = []
            opened_26 += drawn(lambda: result_26.append(draw_cdf_gaps_26(ax_26, cdf_a_26, cdf_b_26, grid_26.copy(), la_26,
                                                                          lb_26, unit_26)))
            why_26 = returned_26(label_26, result_26[0], case_26)
            report_26(not why_26, f"{label_26} : (w1, ks) rendus comme attendu", f"{label_26} : {why_26}")
            missing_26 = look_26(ax_26, cdf_a_26, cdf_b_26, grid_26, la_26, lb_26, unit_26, case_26[2])
            report_26(not missing_26, f"{label_26} : les deux courbes, l'aire, le segment, le titre et la légende sont là",
                      f"{label_26} : il manque " + " ; ".join(missing_26))
        report_26(not why_c26 and not missing_c26, "un appel que tu ne vois pas (deux autres lois, une autre grille) est "
                  "juste aussi", "un appel que tu ne vois pas (deux autres lois, unité « m ») : "
                  + " ; ".join(([why_c26] if why_c26 else []) + (["il manque " + " ; ".join(missing_c26)] if missing_c26 else [])))
        report_26(not opened_26, "draw_cdf_gaps_26 dessine dans l'axe reçu",
                  "draw_cdf_gaps_26 doit dessiner dans l'axe reçu, sans ouvrir de figure")
    except BaseException:
        plt.close(fig_26)
        raise
    fig_26.tight_layout()
    plt.show()
    verdict("M27.26", all(results_26), "tes trois figures sont justes.", "relis les lignes ❌ ci-dessus.")

**Après les figures**, dans ta copie : pourquoi l'écart vertical de la translation est-il le plus grand au milieu ? Pourquoi $W_1$ vaut-il exactement l'écart des moyennes pour la dilatation ? Que mesure KS que $W_1$ ne mesure pas, et inversement ?

## Partie C · Déboguer, tester

Fiche §2 et §8. Trois erreurs classiques, qui passent une vérification mal choisie ; un test de propriété des stratégies optimales d'un jeu $2 \times 2$. Les données et les vérifications s'appuient sur la référence (`ref_distances`), pour ne pas dépendre de ton code.

### Ex M27.27 — Chasse à l'erreur : une stratégie d'indifférence calculée sur ses propres gains, W₂ de deux gaussiennes avec des variances, des échantillons appariés sans être triés 🐛 ★ ⏱️ 25 min
**Objectif :** reconnaître trois erreurs classiques sur l'indifférence, W₂ et l'appariement trié, qui passent une vérification mal choisie, puis les corriger.  
**Prérequis :** fiche §2 et §8 ; Ex M27.2, M27.14, M27.16 et M27.20 · **Parcours :** C

Un collègue a écrit trois fonctions, chacune accompagnée d'une vérification qui passe (cellule fournie) :
1. `penalty_kick_mix(G)` doit rendre $p^\star$, la probabilité optimale de la première ligne d'un jeu $2 \times 2$ sans point selle ; elle rend `(d - b) / (a - b - c + d)`. Il l'a vérifiée sur le jeu $\begin{pmatrix} 50 & 90 \\ 90 & 60 \end{pmatrix}$ : $p$ est entre 0 et 1, et les deux colonnes coûtent alors autant au joueur des colonnes, $50p + 90(1 - p) = 90p + 60(1 - p)$. Il l'applique aux tirs au but de M27.2.
2. `w2_settings(mu1, sd1, mu2, sd2)` doit rendre $W_2$ entre deux réglages d'une machine, résumés par leur moyenne et leur écart-type ; elle rend `math.sqrt((mu1 - mu2) ** 2 + (sd1 ** 2 - sd2 ** 2) ** 2)`. Il l'a vérifiée par `w2_settings(0, 1, 3, 1) == 3` et `w2_settings(2, 0, 2, 1) == 1`.
3. `pair_parts(shafts, sleeves)` doit apparier $n$ axes et $n$ bagues, dont on a mesuré les diamètres (en mm), pour que les jeux bague − axe soient aussi réguliers que possible : elle rend un tableau d'entiers `partner`, où `partner[i]` est l'indice de la bague montée sur l'axe `i`, qui minimise $\sum_i(\text{bague} - \text{axe})^2$ (la fiche, §8, et ∂ M27.14 : trier puis apparier). Elle rend `np.argsort(sleeves)`. Il l'a vérifiée avec les axes `[10.00, 10.02, 10.05]` et les bagues `[10.09, 10.06, 10.11]` : elle rend `[1, 0, 2]`, et les jeux valent 0,06 ; 0,07 ; 0,06.

À faire : trouve et corrige chaque erreur ; les fonctions corrigées s'appellent `penalty_kick_mix_fixed`, `w2_settings_fixed` et `pair_parts_fixed`, avec les mêmes arguments (`pair_parts_fixed` rend aussi un tableau NumPy d'entiers). Puis réponds :
- a) $p^\star$ pour les tirs au but (`G_PENALTY`) par `penalty_kick_mix_fixed`, puis ce que rend `penalty_kick_mix` (une liste de 2 nombres) ;
- b) $W_2$ par `w2_settings_fixed` puis par `w2_settings`, pour la machine du 🔗 M27.16 (`SHAFTS_BEFORE`, `SHAFTS_AFTER`), puis de même pour $\mathcal{N}(100 ;\ 5^2)$ contre $\mathcal{N}(100 ;\ 2^2)$ (une liste de 4 nombres) ;
- c) pour les axes `[20.03, 19.98, 20.01, 19.96, 20.05]` et les bagues `[20.10, 20.04, 20.13, 20.07, 20.06]` (`SHAFTS_27`, `SLEEVES_27`) : ce que rend `pair_parts_fixed`, puis $\sum_i(\text{bague} - \text{axe})^2$ (mm²) pour l'appariement corrigé, puis pour celui du collègue (une liste de 7 : les 5 indices, puis les deux sommes).

Dans ta copie : pour chacune des trois erreurs, pourquoi la vérification du collègue ne la voyait pas. La vérification essaie tes corrections sur des cas que tu ne vois pas : des jeux sans symétrie, des écarts-types différents, des axes dans le désordre et des diamètres égaux.

In [ ]:
SHAFTS_27 = np.array([20.03, 19.98, 20.01, 19.96, 20.05])   # c): the diameters of five shafts (mm)
SLEEVES_27 = np.array([20.10, 20.04, 20.13, 20.07, 20.06])  # and of five sleeves (mm)


# The colleague's code
def penalty_kick_mix(G):
    """p*, the optimal probability of the first row of a 2 × 2 game without a saddle point."""
    (a, b), (c, d) = np.asarray(G, dtype=float).tolist()
    return (d - b) / (a - b - c + d)


def w2_settings(mu1, sd1, mu2, sd2):
    """W2 between two settings of a machine, summed up by their mean and their standard deviation."""
    return math.sqrt((mu1 - mu2) ** 2 + (sd1 ** 2 - sd2 ** 2) ** 2)


def pair_parts(shafts, sleeves):
    """partner[i], the index of the sleeve mounted on the shaft i, so that Σ (sleeve − shaft)² is smallest."""
    return np.argsort(sleeves)


# The colleague's checks: all pass
p_check_27 = penalty_kick_mix([[50, 90], [90, 60]])
print("(1) 0 ≤ p ≤ 1 :", 0 <= p_check_27 <= 1, "; les deux colonnes coûtent autant :",
      math.isclose(50 * p_check_27 + 90 * (1 - p_check_27), 90 * p_check_27 + 60 * (1 - p_check_27)), "(attendu True, True)")
print("(2)", w2_settings(0, 1, 3, 1) == 3, w2_settings(2, 0, 2, 1) == 1, "(attendu True, True)")
shafts_check_27, sleeves_check_27 = np.array([10.00, 10.02, 10.05]), np.array([10.09, 10.06, 10.11])
partner_check_27 = pair_parts(shafts_check_27, sleeves_check_27)
print("(3)", partner_check_27.tolist() == [1, 0, 2], "; jeux :",
      np.round(sleeves_check_27[partner_check_27] - shafts_check_27, 2).tolist(), "(attendu True ; [0.06, 0.07, 0.06])")

In [ ]:
def penalty_kick_mix_fixed(G):
    """p*, the optimal probability of the first row of a 2 × 2 game without a saddle point."""
    raise NotImplementedError("penalty_kick_mix_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def w2_settings_fixed(mu1, sd1, mu2, sd2):
    """W2 between two settings of a machine, summed up by their mean and their standard deviation."""
    raise NotImplementedError("w2_settings_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def pair_parts_fixed(shafts, sleeves):
    """partner[i], the index of the sleeve mounted on the shaft i, so that Σ (sleeve − shaft)² is smallest (an array of
    integers)."""
    raise NotImplementedError("pair_parts_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M27_27a = answer_M27_27b = answer_M27_27c = ...
with wbm.attempt("M27.27"):          # your three functions: ⏳ while they are not written
    answer_M27_27a = ...  # a) a list of 2 numbers, as computed: p* for G_PENALTY by penalty_kick_mix_fixed, then what the WRONG penalty_kick_mix returns
    answer_M27_27b = ...  # b) a list of 4 numbers, as computed: the machine by w2_settings_fixed, then by the WRONG w2_settings; then N(100, 5²) against N(100, 2²), the same two ways
    answer_M27_27c = ...  # c) a list of 7: the 5 indices of pair_parts_fixed(SHAFTS_27, SLEEVES_27), then Σ (sleeve − shaft)² for your pairing, then for the colleague's

In [ ]:
for letter, answer in zip("abc", [answer_M27_27a, answer_M27_27b, answer_M27_27c]):
    check_answer(f"M27.27{letter}", answer)

CASES_C27 = {'kicks': [{'G': [[2.0, 7.0], [6.0, 3.0]], 'want': 0.375, 'q': 0.5},
           {'G': [[9.0, 0.0], [2.0, 5.0]], 'want': 0.25, 'q': 0.4166666666666667},
           {'G': [[-4.0, 5.0], [3.0, -1.0]], 'want': 0.3076923076923077, 'q': 0.46153846153846156}],
 'settings': [{'args': [5.0, 0.3, 5.4, 0.8], 'want': 0.6403124237432851, 'variances': 0.6800735254367725},
              {'args': [-2.0, 4.0, 1.0, 1.5], 'want': 3.905124837953327, 'variances': 14.073467945037569},
              {'args': [0.0, 0.5, 0.0, 2.5], 'want': 2.0, 'variances': 6.0}],
 'parts': [{'shafts': [30.05, 29.96, 29.98, 29.99, 30.03, 30.02],
            'sleeves': [30.07, 30.06, 30.01, 30.11, 30.12, 30.08],
            'best': 0.030199999999999706,
            'sleeves_only': [2, 1, 0, 5, 3, 4],
            'composed': [1, 0, 5, 4, 3, 2]},
           {'shafts': [29.93, 29.99, 30.02, 29.95, 29.96, 29.98, 30.03],
            'sleeves': [30.11, 30.09, 30.1, 30.05, 30.04, 30.14, 30.06],
            'best': 0.07650000000000011,
            'sleeves_only': [4, 3, 6, 1, 2, 0, 5],
            'composed': [4, 1, 2, 0, 3, 6, 5]},
           {'shafts': [5.0, 5.0, 4.98, 5.02],
            'sleeves': [5.06, 5.05, 5.06, 5.04],
            'best': 0.011299999999999892,
            'sleeves_only': [3, 1, 0, 2],
            'composed': [0, 3, 1, 2]}]}   # the cases you do not see, with what each corrected function must give

results_27, parts_27 = [], []


def report_27(ok, success, failure):
    results_27.append(bool(ok))
    print(("✅ " + success) if ok else ("❌ " + failure))


def call_27(part, name, function, *args):
    """function(*args); an error of the learner's code becomes one ❌ line that names the function (the check of this
    part stops there, the final verdict stays)."""
    try:
        return function(*args)
    except (NotImplementedError, MymathMissingError):
        raise
    except Exception as exc:  # noqa: BLE001 - the learner's error, reported with the name of the function
        report_27(False, "", f"({part}) {name} : ton code lève {type(exc).__name__} : {str(exc)[:200]}")
        parts_27.append(part)
        raise StopCheck from None


with checked("M27.27"):                                       # (1) penalty_kick_mix_fixed
    wrong_27 = []
    for case_c27 in CASES_C27["kicks"]:
        got_c27 = call_27(1, "penalty_kick_mix_fixed", penalty_kick_mix_fixed, np.array(case_c27["G"]))
        try:
            value_c27 = float(got_c27)
        except (TypeError, ValueError):
            value_c27 = float("nan")
        if math.isclose(value_c27, case_c27["want"], rel_tol=1e-9):
            continue
        if math.isclose(value_c27, case_c27["q"], rel_tol=1e-9):
            wrong_27.append("ta fonction rend encore la probabilité qui rend le joueur des LIGNES indifférent ; p* rend "
                            "l'adversaire indifférent (ce que coûtent les deux colonnes)")
        elif math.isclose(value_c27, 1 - case_c27["want"], rel_tol=1e-9):
            wrong_27.append("p* est la probabilité de la première ligne, pas de la seconde")
        else:
            wrong_27.append(f"un jeu sans symétrie : attendu {case_c27['want']:.6f}, reçu {shown(got_c27)}")
    report_27(not wrong_27, "(1) penalty_kick_mix_fixed : trois jeux que tu ne vois pas, sans symétrie",
              "(1) penalty_kick_mix_fixed : " + " ; ".join(list(dict.fromkeys(wrong_27))[:2]))
    parts_27.append(1)

with checked("M27.27"):                                       # (2) w2_settings_fixed
    wrong_27 = []
    for case_c27 in CASES_C27["settings"]:
        got_c27 = call_27(2, "w2_settings_fixed", w2_settings_fixed, *case_c27["args"])
        try:
            value_c27 = float(got_c27)
        except (TypeError, ValueError):
            value_c27 = float("nan")
        if math.isclose(value_c27, case_c27["want"], rel_tol=1e-9):
            continue
        if math.isclose(value_c27, case_c27["variances"], rel_tol=1e-9):
            wrong_27.append("tu mets encore des variances : la formule soustrait les écarts-types")
        elif math.isclose(value_c27, case_c27["want"] ** 2, rel_tol=1e-9):
            wrong_27.append("ta fonction rend W₂², pas W₂")
        else:
            wrong_27.append(f"(moyenne, écart-type) = {case_c27['args'][:2]} et {case_c27['args'][2:]} : attendu "
                            f"{case_c27['want']:.6f}, reçu {shown(got_c27)}")
    report_27(not wrong_27, "(2) w2_settings_fixed : trois réglages que tu ne vois pas, aux écarts-types différents",
              "(2) w2_settings_fixed : " + " ; ".join(list(dict.fromkeys(wrong_27))[:2]))
    parts_27.append(2)

with checked("M27.27"):                                       # (3) pair_parts_fixed
    wrong_27 = []
    for case_c27 in CASES_C27["parts"]:
        shafts_c27, sleeves_c27 = np.array(case_c27["shafts"]), np.array(case_c27["sleeves"])
        got_c27 = call_27(3, "pair_parts_fixed", pair_parts_fixed, shafts_c27.copy(), sleeves_c27.copy())
        arr_c27 = np.asarray(got_c27)
        n_c27 = len(shafts_c27)
        if arr_c27.shape != (n_c27,) or not np.issubdtype(arr_c27.dtype, np.integer):
            wrong_27.append(f"pair_parts_fixed doit rendre un tableau NumPy d'entiers de {n_c27} indices, reçu "
                            f"{shown(got_c27)[:120]} (dtype {getattr(arr_c27, 'dtype', '?')})")
            continue
        if sorted(arr_c27.tolist()) != list(range(n_c27)):
            wrong_27.append("chaque bague est montée une fois : partner doit contenir chacun des indices 0, …, n − 1")
            continue
        total_c27 = float(((sleeves_c27[arr_c27] - shafts_c27) ** 2).sum())
        if total_c27 <= case_c27["best"] + 1e-12:
            continue
        if arr_c27.tolist() == case_c27["sleeves_only"]:
            wrong_27.append("tu ne ranges encore que les bagues : la k-ième plus petite bague va sur le k-ième plus petit "
                            "axe, quel que soit l'ordre des mesures")
        elif arr_c27.tolist() == case_c27["composed"]:
            wrong_27.append("partner[i] est la bague de l'axe i : la k-ième plus petite bague va sur l'axe de rang k "
                            "(l'indice argsort(shafts)[k]), pas sur l'axe d'indice k")
        else:
            wrong_27.append(f"{n_c27} axes dans le désordre : ta somme des carrés vaut {total_c27:.6f}, la plus petite "
                            f"{case_c27['best']:.6f}")
    report_27(not wrong_27, "(3) pair_parts_fixed : des axes dans le désordre, et des diamètres égaux",
              "(3) pair_parts_fixed : " + " ; ".join(list(dict.fromkeys(wrong_27))[:2]))
    parts_27.append(3)

if len(set(parts_27)) == 3:
    verdict("M27.27", all(results_27), "les trois fonctions corrigées passent tous les cas.", "relis les lignes ❌ ci-dessus.")

**Après la vérification**, dans ta copie : pour chaque erreur, pourquoi la vérification du collègue ne la voyait-elle pas (un jeu symétrique, des écarts-types égaux ou valant 0 et 1, des axes déjà triés), et quelle vérification l'aurait vue ?

### Ex M27.28 — Un test de propriété : la valeur d'un jeu 2 × 2 tiré au hasard est entre maximin et minimax, et devient v + c pour G + c et k v pour k G (k > 0) 🛠️ ★ ⏱️ 20 min
**Objectif :** écrire un test de propriété qui compte ses échecs sur des jeux tirés au hasard, voir que les propriétés du titre laissent passer des versions fausses, et en ajouter deux qui les attrapent.  
**Prérequis :** fiche §2 ; Ex M27.20 · **Parcours :** C

La cellule fournie donne quatre versions fausses de `mixed_strategy_value_2x2` : `swapped_pq` (elle rend $(q, p, v)$), `formulas_always` (les formules d'indifférence même quand il y a un point selle, sauf si $\kappa = 0$), `maximin_as_value` (les bonnes stratégies, mais le maximin comme valeur) et `rounded_value` (la valeur arrondie à l'entier).

Écris `game_property_test_28(n_trials, rng, solver=None, tol=1e-9)` ; quand `solver` vaut `None`, le test prend `mymath.distances.mixed_strategy_value_2x2` **au moment de l'appel**. Pour chacun des `n_trials` essais, dans cet ordre : `G = rng.integers(-9, 10, size=(2, 2)).astype(float)`, `c = float(rng.integers(-5, 6))`, `k = float(rng.integers(2, 6))` ; puis `p, q, v = solver(G)`, et le maximin et le minimax de `G`, calculés dans le test par `G.min(axis=1).max()` et `G.max(axis=0).min()`, sans appeler `mymath`. La fonction rend un dictionnaire de cinq compteurs d'essais (un essai compte une fois s'il échoue) :
- `"bounds"` : $v$ sort de $[\text{maximin} - \text{tol} ;\ \text{minimax} + \text{tol}]$, ou $p$ ou $q$ sort de $[-\text{tol} ;\ 1 + \text{tol}]$ ;
- `"shift"` : `solver(G + c)` s'écarte de $(p, q, v + c)$ de plus de `tol` en une coordonnée ;
- `"scale"` : `solver(k * G)` s'écarte de $(p, q, kv)$ de plus de `tol` en une coordonnée ;
- et, **seulement pour les jeux sans point selle** (maximin $<$ minimax), `"transpose"` : `solver(-G.T)` s'écarte de $(q, p, -v)$, et `"swap_rows"` : `solver(G[::-1])` s'écarte de $(1 - p, q, v)$.

Puis :
- a) Dans ta copie : pourquoi des gains entiers, et des $c$ et $k$ entiers ? Pourquoi les deux dernières propriétés seulement sans point selle (essaie $\mathbf{G} = \begin{pmatrix} 2 & 2 \\ 2 & 2 \end{pmatrix}$) ?
- b) Avec `rng = np.random.default_rng(6428)` et 400 essais, avec ta fonction : vrai ou faux, les cinq compteurs sont nuls ;
- c) avec le même départ (un nouveau `default_rng(6428)` chaque fois, 400 essais) et chacune des quatre versions fausses, dans l'ordre `swapped_pq`, `formulas_always`, `maximin_as_value`, `rounded_value` : les cinq compteurs, dans l'ordre `"bounds"`, `"shift"`, `"scale"`, `"transpose"`, `"swap_rows"` (une liste de 4 listes de 5 entiers) ;
- d) les noms des versions fausses qui passent les trois propriétés du titre (`"bounds"`, `"shift"` et `"scale"` nuls) (une liste de noms, dans n'importe quel ordre, comme `["formulas_always"]`).

La vérification lance ton test sur deux départs que tu ne vois pas (150 essais chacun) : avec ta fonction, où ses cinq compteurs doivent valoir ceux de la référence (0), puis avec chacune des quatre versions fausses, où ils doivent valoir **exactement** ceux de la référence. Elle remplace aussi un instant `mymath.distances.mixed_strategy_value_2x2` par une version fausse (une valeur calculée avec $\lvert\kappa\rvert$) : tes compteurs doivent alors compter, comme ceux de la référence. Chacune des cinq propriétés est ainsi exigée par au moins une version fausse. Un entier NumPy convient pour un compteur ; un test qui échoue ne lève pas d'`AssertionError` : il compte. Ce qu'il faut retenir : les propriétés du titre, déjà vérifiées par `tests/test_m27_distances.py`, laissent passer des versions fausses ; l'échange des rôles et l'échange des lignes, qui ne valent que si les stratégies optimales sont uniques, les attrapent ; un test se juge aux erreurs qu'il attrape.

In [ ]:
def swapped_pq(G):
    """WRONG: (q, p, v) instead of (p, q, v) (built on the reference)."""
    p, q, v = ref_distances.mixed_strategy_value_2x2(G)
    return q, p, v


def formulas_always(G):
    """WRONG: the indifference formulas even when there is a saddle point (the reference only when κ = 0)."""
    (a, b), (c, d) = np.asarray(G, dtype=float).tolist()
    kappa = a - b - c + d
    if kappa == 0:
        return ref_distances.mixed_strategy_value_2x2(G)
    return (d - c) / kappa, (d - b) / kappa, (a * d - b * c) / kappa


def maximin_as_value(G):
    """WRONG: the right strategies, but the maximin as value."""
    p, q, v = ref_distances.mixed_strategy_value_2x2(G)
    return p, q, ref_distances.pure_strategy_values(G)[0]


def rounded_value(G):
    """WRONG: the value rounded to an integer (Python's round: to the even integer on a tie, round(2.5) = 2)."""
    p, q, v = ref_distances.mixed_strategy_value_2x2(G)
    return p, q, float(round(v))

In [ ]:
def game_property_test_28(n_trials, rng, solver=None, tol=1e-9):
    """n_trials random games: G = rng.integers(-9, 10, size=(2, 2)).astype(float), then c = float(rng.integers(-5, 6)),
    then k = float(rng.integers(2, 6)); solver defaults to mymath.distances.mixed_strategy_value_2x2, looked up at the
    call; the maximin and the minimax of G by G.min(axis=1).max() and G.max(axis=0).min(). Returns the numbers of trials
    "bounds", "shift", "scale", "transpose", "swap_rows" that fail (the last two only for games without a saddle point)."""
    raise NotImplementedError("game_property_test_28 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M27_28b = answer_M27_28c = answer_M27_28d = ...
with wbm.attempt("M27.28"):          # your test and your library: ⏳ while they are not written
    answer_M27_28b = ...  # b) True or False: the five counters are 0 (default_rng(6428), 400 trials, your function)
    answer_M27_28c = ...  # c) a list of 4 lists of 5 integers: the counters ("bounds", "shift", "scale", "transpose", "swap_rows") for swapped_pq, formulas_always, maximin_as_value, rounded_value (a new default_rng(6428) each time, 400 trials)
    answer_M27_28d = ...  # d) a list of names (strings, any order): the wrong versions whose "bounds", "shift" and "scale" are 0

In [ ]:
for letter, answer in zip("bcd", [answer_M27_28b, answer_M27_28c, answer_M27_28d]):
    check_answer(f"M27.28{letter}", answer)

def absolute_kappa_c28(G):
    """WRONG: without a saddle point, the value (ad − bc)/|κ| (the sign of κ lost)."""
    p, q, v = ref_distances.mixed_strategy_value_2x2(G)
    maximin, minimax, _, _ = ref_distances.pure_strategy_values(G)
    if maximin == minimax:
        return p, q, v
    (a, b), (c, d) = np.asarray(G, dtype=float).tolist()
    return p, q, (a * d - b * c) / abs(a - b - c + d)


with checked("M27.28"):
    results_28 = []
    report_28 = reporter(results_28, stop=False)
    COUNTERS_28 = ('bounds', 'shift', 'scale', 'transpose', 'swap_rows')
    EXPECTED_C28 = {"6441": {"ref": {"bounds": 0, "shift": 0, "scale": 0, "transpose": 0, "swap_rows": 0}, "swapped_pq": {"bounds": 0, "shift": 0, "scale": 0, "transpose": 0, "swap_rows": 40}, "formulas_always": {"bounds": 93, "shift": 0, "scale": 0, "transpose": 0, "swap_rows": 0}, "maximin_as_value": {"bounds": 0, "shift": 0, "scale": 0, "transpose": 40, "swap_rows": 0}, "rounded_value": {"bounds": 0, "shift": 3, "scale": 31, "transpose": 0, "swap_rows": 0}, "absolute_kappa_c28": {"bounds": 9, "shift": 13, "scale": 0, "transpose": 39, "swap_rows": 39}}, "6442": {"ref": {"bounds": 0, "shift": 0, "scale": 0, "transpose": 0, "swap_rows": 0}, "swapped_pq": {"bounds": 0, "shift": 0, "scale": 0, "transpose": 0, "swap_rows": 40}, "formulas_always": {"bounds": 91, "shift": 0, "scale": 0, "transpose": 0, "swap_rows": 0}, "maximin_as_value": {"bounds": 0, "shift": 0, "scale": 0, "transpose": 41, "swap_rows": 0}, "rounded_value": {"bounds": 0, "shift": 2, "scale": 31, "transpose": 0, "swap_rows": 0}}}   # the reference, per hidden start
    PROPERTIES_C28 = {
        "bounds": "que v est entre maximin − tol et minimax + tol, et p, q entre −tol et 1 + tol",
        "shift": "que solver(G + c) redonne (p, q, v + c) à tol près",
        "scale": "que solver(k G) redonne (p, q, k v) à tol près",
        "transpose": "que solver(−Gᵀ) redonne (q, p, −v), sans point selle",
        "swap_rows": "que solver(G[::-1]) redonne (1 − p, q, v), sans point selle",
    }

    def run_c28(label, seed, **kwargs):
        """Your game_property_test_28 on the hidden start seed (150 trials): its dictionary."""
        try:
            counts = game_property_test_28(150, np.random.default_rng(seed), **kwargs)
        except (NotImplementedError, MymathMissingError):
            raise
        except AssertionError as exc:
            print(f"❌ Ex M27.28 : game_property_test_28 ({label}) lève AssertionError ({str(exc)[:120]}) : l'erreur sort de "
                  "ta fonction ; elle doit compter les essais qui échouent et rendre le dictionnaire, sans assert.")
            raise StopCheck from None
        except Exception as exc:  # noqa: BLE001 - the learner's error
            print(f"❌ Ex M27.28 : game_property_test_28 ({label}) lève {type(exc).__name__} : {str(exc)[:200]}")
            raise StopCheck from None
        if not isinstance(counts, dict) or set(counts) != set(COUNTERS_28):
            print("❌ Ex M27.28 : game_property_test_28 doit rendre un dictionnaire de clés " + ", ".join(COUNTERS_28)
                  + f", reçu {shown(counts)[:160]}")
            raise StopCheck
        for key in COUNTERS_28:
            value = counts[key]
            if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)):
                print(f"❌ Ex M27.28 : le compteur \"{key}\" doit être un nombre entier d'essais (un entier NumPy convient), "
                      f"reçu {type(value).__name__} {value!r}")
                raise StopCheck
        return {key: int(v) for key, v in counts.items()}

    def compare_c28(label, got, want):
        """One ✅ when the five counters equal the reference's; else one ❌ per counter, naming the property."""
        wrong = [k for k in COUNTERS_28 if got[k] != want[k]]
        if not wrong:
            seen = [f"\"{k}\" {want[k]}" for k in COUNTERS_28 if want[k]]
            report_28(True, f"{label} : les cinq compteurs valent ceux de la référence"
                      + (" (" + ", ".join(seen) + ", les autres nuls)" if seen else " (0)"), "")
        for k in wrong:
            if got[k] == 0:
                why = f"ton test ne vérifie pas {PROPERTIES_C28[k]}"
            elif want[k] == 0:
                why = ("cette version ne fausse pas cette propriété : ton contrôle compte trop (sa tolérance, ce qu'il "
                       "compare, ou les jeux sur lesquels il porte)")
            else:
                why = ("un essai compte une fois s'il échoue ; les deux dernières propriétés ne portent que sur les jeux "
                       "sans point selle")
            report_28(False, "", f"{label} : \"{k}\" = {got[k]}, la référence {want[k]} : {why}")

    for seed_c28 in (6441, 6442):
        expected_c28 = EXPECTED_C28[str(seed_c28)]
        mine_c28 = run_c28(f"départ caché {seed_c28}", seed_c28)
        wrong_c28 = [k for k in COUNTERS_28 if mine_c28[k] != expected_c28["ref"][k]]
        report_28(not wrong_c28, f"départ caché {seed_c28}, ta fonction : les cinq compteurs valent ceux de la référence (0)",
                  f"départ caché {seed_c28}, ta fonction : " + ", ".join(f"{k} = {mine_c28[k]}" for k in wrong_c28)
                  + " au lieu de 0 : vérifie l'ordre des tirages (G, puis c, puis k), tes tolérances, puis ta "
                  "mixed_strategy_value_2x2 (ses tests, M27.20)")
        for name_c28, function_c28 in (("swapped_pq", swapped_pq), ("formulas_always", formulas_always),
                                       ("maximin_as_value", maximin_as_value), ("rounded_value", rounded_value)):
            got_c28 = run_c28(f"{name_c28}, départ caché {seed_c28}", seed_c28, solver=function_c28)
            compare_c28(f"{name_c28}, départ caché {seed_c28}", got_c28, expected_c28[name_c28])

    original_c28 = mymath.distances.mixed_strategy_value_2x2
    aliases_c28 = [alias for alias, value in list(globals().items()) if value is original_c28]
    calls_c28 = [0]

    def replaced_c28(G):
        calls_c28[0] += 1
        return absolute_kappa_c28(G)

    mymath.distances.mixed_strategy_value_2x2 = replaced_c28
    for alias_c28 in aliases_c28:
        globals()[alias_c28] = replaced_c28
    try:
        patched_c28 = run_c28("une valeur calculée avec |κ|", 6441)
    finally:
        mymath.distances.mixed_strategy_value_2x2 = original_c28
        for alias_c28 in aliases_c28:
            globals()[alias_c28] = original_c28
    if calls_c28[0] == 0:
        report_28(False, "", "une version fausse remplace un instant ta mixed_strategy_value_2x2, mais ton test ne l'appelle "
                  "jamais : avec solver=None, il doit prendre mymath.distances.mixed_strategy_value_2x2 au moment de "
                  "l'appel, sans en garder une copie")
    else:
        compare_c28("une valeur calculée avec |κ| à la place de ta mixed_strategy_value_2x2", patched_c28,
                    EXPECTED_C28["6441"]["absolute_kappa_c28"])
    verdict("M27.28", all(results_28), "ton test compte juste sur la référence et sur les versions fausses, et prend ta "
            "fonction au moment de l'appel.", "relis les lignes ❌ ci-dessus.")

**Dans ta copie** : a) pourquoi des gains entiers, et des $c$ et $k$ entiers ; pourquoi les deux dernières propriétés seulement sans point selle.

**Après la vérification**, dans ta copie : pourquoi `swapped_pq` et `maximin_as_value` passent-elles les trois propriétés du titre ? Quelles propriétés, prises ensemble, caractérisent la solution d'un jeu $2 \times 2$ sans point selle ?

## Partie D · Défi

Fiche §8. La loi normale la plus proche d'une loi exponentielle au sens de W₂, calculée à partir de sa seule fonction quantile, et jugée sur les valeurs exactes de trois lois.

### Ex M27.29 — Défi : la gaussienne la plus proche d'une loi exponentielle au sens de W₂, et cette distance, à 10⁻³ près 🏆 ★★★ ⏱️ 30 min
**Objectif :** trouver la loi normale la plus proche d'une loi donnée par sa fonction quantile, au sens de W₂, et la distance qui reste, à 10⁻³ près.  
**Prérequis :** fiche §8 ; M02a, M02b ; Ex M27.8 et M27.22 · **Parcours :** C

Le temps d'attente à un guichet suit la loi exponentielle de moyenne 2 minutes : $F(x) = 1 - e^{-x/2}$ pour $x \ge 0$, de fonction quantile $F^{-1}(u) = -2\ln(1 - u)$ (`counter_quantile_29`) et de variance 4 (`VARIANCE_29`). On cherche la loi normale $\mathcal{N}(\mu, \sigma^2)$, $\sigma \ge 0$, la plus proche de cette loi au sens de $W_2$, et cette distance.

Écris `closest_gaussian_29(quantile)`, qui reçoit la fonction quantile d'une loi de moment d'ordre 2 fini (une fonction NumPy de $u \in\ ]0, 1[$, qui accepte un tableau) et rend `(mu, sigma, w2)`. Tout NumPy et SciPy sont permis (`scipy.integrate.quad`, `scipy.optimize`, `scipy.stats.norm.ppf`), mais la fonction ne reçoit que la fonction quantile : aucune formule propre à la loi exponentielle.

**Critère de réussite :** `mu`, `sigma` et `w2` à moins de $10^{-3}$ des valeurs exactes pour l'attente au guichet, et pour deux lois de la vérification que tu ne vois pas, en moins de 10 secondes par loi. La vérification refuse une fonction qui rend les mêmes nombres pour trois lois différentes, et reconnaît une variance à la place d'un écart-type, un carré à la place de $W_2$, l'écart-type de la loi à la place du meilleur $\sigma$, et une intégrale trop grossière. Relève :
- a) `(mu, sigma, w2)` pour l'attente au guichet (un tuple de 3, tel que rendu ; 3 décimales dans ta copie) ;
- b) vrai ou faux : $\mu$ est la moyenne de la loi exponentielle ;
- c) $W_2^2$ divisé par la variance de la loi exponentielle (4) : la part de la variance que la meilleure gaussienne ne peut pas rendre (un nombre, en %, tel que calculé ; 1 décimale dans ta copie).

Point de départ : $W_2^2(\mu, \sigma) = \int_0^1\big(F^{-1}(u) - \mu - \sigma\Phi^{-1}(u)\big)^2\,\mathrm{d}u$ (fiche §8) ; une somme sur une grille régulière de $u$ converge lentement, à cause des queues.

In [ ]:
def counter_quantile_29(u):
    """The quantile function of the wait at the counter, the exponential law of mean 2 minutes: F⁻¹(u) = −2 ln(1 − u)."""
    return -2.0 * np.log1p(-np.asarray(u, dtype=float))


VARIANCE_29 = 4.0                                          # the variance of this law (min²)

In [ ]:
def closest_gaussian_29(quantile):
    """The normal law N(mu, sigma²), sigma >= 0, closest in W2 to the law of quantile function quantile (a NumPy function
    of u in ]0, 1[ that accepts an array), and this distance: returns (mu, sigma, w2). No formula of a particular law."""
    raise NotImplementedError("closest_gaussian_29 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M27_29a = answer_M27_29b = answer_M27_29c = ...
with wbm.attempt("M27.29"):          # your function: ⏳ while it is not written
    answer_M27_29a = ...  # a) a tuple of 3, as returned: (mu, sigma, w2) for the wait at the counter
    answer_M27_29b = ...  # b) True or False: mu is the mean of the exponential law
    answer_M27_29c = ...  # c) a number, as computed: 100 W2² / 4, in %

In [ ]:
for letter, answer in zip("abc", [answer_M27_29a, answer_M27_29b, answer_M27_29c]):
    check_answer(f"M27.29{letter}", answer)

LAWS_C29 = [([2.000000000000009, 1.806394568729293, 0.8584513109418084], 2.0),
 ([2.5, 0.8462843753173325, 0.18385525852273368], 0.8660254037844387),
 ([2.999999999999689, 2.010791092025536, 0.6758100204814919], 2.121320343559643)]   # per law: the exact (mu, sigma, w2), and the standard deviation of the law

with checked("M27.29"):
    results_29 = []
    report_29 = reporter(results_29, stop=False)
    laws_29 = (("l'attente au guichet", counter_quantile_29),
               ("une loi uniforme que tu ne vois pas", lambda u: 1.0 + 3.0 * np.asarray(u, dtype=float)),
               ("une loi Gamma que tu ne vois pas", stats.gamma(2, scale=1.5).ppf))
    runs_29 = []
    for (label_29, quantile_29), law_29 in zip(laws_29, LAWS_C29):
        start_29 = time.perf_counter()
        result_29 = closest_gaussian_29(quantile_29)
        elapsed_29 = time.perf_counter() - start_29
        triple_29 = pair("M27.29", "closest_gaussian_29", result_29, names=("mu", "sigma", "w2"))
        try:
            triple_29 = tuple(float(v) for v in triple_29)
        except (TypeError, ValueError):
            verdict("M27.29", False, "", f"closest_gaussian_29 doit rendre trois nombres (mu, sigma, w2), reçu {shown(result_29)}")
            raise StopCheck from None
        runs_29.append((label_29, triple_29, elapsed_29, law_29))
    if len({run[1] for run in runs_29}) == 1:
        verdict("M27.29", False, "", "closest_gaussian_29 rend les mêmes nombres pour trois lois différentes : elle doit les "
                "calculer à partir de la fonction quantile reçue")
        raise StopCheck
    for k_29, (label_29, (mu_29, sigma_29, w2_29), elapsed_29, law_29) in enumerate(runs_29):
        (mu_e29, sigma_e29, w2_e29), sd_29 = law_29
        close_29 = [abs(got - want) < 1e-3 for got, want in ((mu_29, mu_e29), (sigma_29, sigma_e29), (w2_29, w2_e29))]
        if all(close_29):
            why_29 = ""
        elif k_29 > 0 and all(abs(a - b) < 1e-9 for a, b in zip((mu_29, sigma_29, w2_29), runs_29[0][1])):
            why_29 = "tu rends les nombres de l'attente au guichet : calcule-les à partir de la fonction quantile reçue"
        elif abs(sigma_29 - sigma_e29 ** 2) < 1e-3 * max(1.0, sigma_e29 ** 2):
            why_29 = "sigma est un écart-type, pas une variance"
        elif close_29[0] and close_29[1] and abs(w2_29 - w2_e29 ** 2) < 1e-3:
            why_29 = "w2 est la distance W₂, pas son carré"
        elif abs(sigma_29 - sd_29) < 1e-3:
            why_29 = ("ton sigma est l'écart-type de la loi : il ne minimise pas W₂ ; écris W₂²(μ, σ) et annule sa dérivée "
                      "en σ")
        elif all(abs(got - want) < 1e-2 for got, want in ((mu_29, mu_e29), (sigma_29, sigma_e29), (w2_29, w2_e29))):
            why_29 = ("à plus de 10⁻³ des valeurs exactes, mais à moins de 10⁻² : une somme sur une grille régulière de u "
                      "converge trop lentement à cause des queues (scipy.integrate.quad, ou une grille resserrée vers 0 et 1)")
        else:
            names_29 = [name for name, ok in zip(("mu", "sigma", "w2"), close_29) if not ok]
            why_29 = ", ".join(names_29) + " à plus de 10⁻³ de la valeur exacte"
            if k_29 > 0:
                why_29 += f" (attendu ≈ ({mu_e29:.4f}, {sigma_e29:.4f}, {w2_e29:.4f}), reçu ({mu_29:.4f}, {sigma_29:.4f}, {w2_29:.4f}))"
        report_29(not why_29 and elapsed_29 < 10, f"{label_29} : (mu, sigma, w2) à moins de 10⁻³ des valeurs exactes, en "
                  f"{fr(elapsed_29, 2)} s", f"{label_29} : " + (why_29 or f"trop lent ({fr(elapsed_29, 1)} s, il faut moins de "
                                                                       "10 s)"))
    verdict("M27.29", all(results_29), "critère atteint : les trois lois à 10⁻³ près, en moins de 10 secondes chacune.",
            "critère pas encore atteint : relis les lignes ❌ ci-dessus.")

**Dans ta copie** : a) avec 3 décimales, et ta méthode ; pourquoi la meilleure gaussienne ne peut-elle pas rendre toute la variance de la loi exponentielle ?

**Pour réfléchir** : pourquoi $\sigma$ est-il plus petit que l'écart-type de la loi (2) ? Que vaudrait $W_2$ pour une loi normale ? Que devient la part de c) pour une loi uniforme ?

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu trouver le maximin, le minimax et les points selles d'une matrice de gains, et réduire un jeu par dominance ?
2. Sais-tu calculer les stratégies optimales et la valeur d'un jeu $2 \times 2$, avec ou sans point selle ?
3. Sais-tu prévoir par les valeurs propres d'un pas ce que fait la descente-montée sur un jeu bilinéaire ?
4. Sais-tu calculer un plan de transport, $W_1$ et $W_2$ en dimension 1, et $W_2$ de deux gaussiennes ?
5. Sais-tu dire pourquoi une distance estimée sur des échantillons est biaisée, et ce que mesurent KS et $W_1$ ?

**La suite** : M28, sur la matrice de Gram, la montée sur l'entrée et les lois tronquées, puis le chapitre 27 de workbookIA, qui applique ces outils aux réseaux génératifs adverses.